# **Downloading the Data**

In [1]:
import requests
from pathlib import Path

save_folder = Path("data/tennis_atp")
save_folder.mkdir(exist_ok = True)

for year in range(1991,2027):
    file_types = ["atp_matches", "atp_matches_futures", "atp_matches_qual_chall"]

    for file_type in file_types:
        file_name = f"{file_type}_{year}.csv"

        url = f"https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/{file_name}"

        try:
            response = requests.get(url)

            if response.status_code == 200:
                with open(save_folder / file_name, "wb") as f:
                    f.write(response.content)
                    print(f"Downloaded {file_name}")

            else:
                print(f"Missing {file_name}")

        
        except Exception as e:
            print(e)

Downloaded atp_matches_1991.csv
Downloaded atp_matches_futures_1991.csv
Downloaded atp_matches_qual_chall_1991.csv
Downloaded atp_matches_1992.csv
Downloaded atp_matches_futures_1992.csv
Downloaded atp_matches_qual_chall_1992.csv
Downloaded atp_matches_1993.csv
Downloaded atp_matches_futures_1993.csv
Downloaded atp_matches_qual_chall_1993.csv
Downloaded atp_matches_1994.csv
Downloaded atp_matches_futures_1994.csv
Downloaded atp_matches_qual_chall_1994.csv
Downloaded atp_matches_1995.csv
Downloaded atp_matches_futures_1995.csv
Downloaded atp_matches_qual_chall_1995.csv
Downloaded atp_matches_1996.csv
Downloaded atp_matches_futures_1996.csv
Downloaded atp_matches_qual_chall_1996.csv
Downloaded atp_matches_1997.csv
Downloaded atp_matches_futures_1997.csv
Downloaded atp_matches_qual_chall_1997.csv
Downloaded atp_matches_1998.csv
Downloaded atp_matches_futures_1998.csv
Downloaded atp_matches_qual_chall_1998.csv
Downloaded atp_matches_1999.csv
Downloaded atp_matches_futures_1999.csv
Download

# **Deduplication and Converting to Dataframe**

In [18]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data/tennis_atp")

FILE_TYPES = ["atp_matches", "atp_matches_futures", "atp_matches_qual_chall"]

dfs = []
loaded_sources = []

for year in range(1991, 2027):
    for family in FILE_TYPES:
        file_name = f"{family}_{year}.csv"
        file_path = DATA_DIR / file_name

        if not file_path.exists():
            print(f"Skipping missing file: {file_name}")
            continue

        df = pd.read_csv(file_path)

        # Preserve source information
        df["_source_file"] = file_name
        df["_source_family"] = family
        df["_source_row"] = np.arange(len(df)) + 2

        loaded_sources.append((file_name, len(df)))
        dfs.append(df)

assert dfs, "No input CSVs were loaded."

matches_df = pd.concat(dfs, ignore_index=True, sort=False)

assert len(matches_df) == sum(count for _, count in loaded_sources)

assert not matches_df[["_source_file", "_source_row"]].duplicated().any()

print("Files loaded:", len(loaded_sources))
print("Total records:", len(matches_df))

Files loaded: 108
Total records: 855194


In [19]:
# Original columns, excluding provenance
RAW_COLUMNS = [
    c for c in matches_df.columns
    if not c.startswith("_source_")
]

REQUIRED = [
    "tourney_id", "tourney_name", "tourney_date",
    "round", "match_num", "winner_id",
    "loser_id", "score"
]

assert set(REQUIRED).issubset(matches_df.columns)

def normalize_text(series):
    return (
        series.astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
        .replace("", pd.NA)
    )

# Keep detection fields separate from matches_df
dedup_view = pd.DataFrame(index=matches_df.index)

dedup_view["event"] = normalize_text(matches_df["tourney_id"]).str.upper()

dedup_view["round"] = normalize_text(matches_df["round"]).str.upper()

dedup_view["match_no"] = (
    normalize_text(matches_df["match_num"])
    .str.replace(r"\.0+$", "", regex=True)
)

w = (
    normalize_text(matches_df["winner_id"])
    .str.replace(r"\.0+$", "", regex=True)
)

l = (
    normalize_text(matches_df["loser_id"])
    .str.replace(r"\.0+$", "", regex=True)
)

# Unordered player pairing
flip = w.fillna("").gt(l.fillna(""))

dedup_view["player_1"] = w.where(~flip, l)
dedup_view["player_2"] = l.where(~flip, w)

dedup_view["winner"] = w
dedup_view["score"] = normalize_text(
    matches_df["score"]
).str.upper()

dedup_view["date"] = normalize_text(
    matches_df["tourney_date"]
)

dedup_view["name"] = normalize_text(
    matches_df["tourney_name"]
).str.upper()

dedup_view["source_family"] = matches_df["_source_family"]
dedup_view["source_file"] = matches_df["_source_file"]

CORE_KEY = [
    "event", "round", "player_1",
    "player_2", "match_no"
]

EVENT_PAIR_KEY = [
    "event", "round", "player_1", "player_2"
]

dedup_view["valid_identity"] = (
    dedup_view[CORE_KEY].notna().all(axis=1)
    & w.ne(l).fillna(False)
)


In [20]:
# 1. Exact duplicate rows
exact_mask = matches_df.duplicated(
    subset=RAW_COLUMNS, keep=False
)

exact_excess = matches_df.duplicated(
    subset=RAW_COLUMNS, keep="first"
).sum()

# 2. Strong identity collisions
valid = dedup_view["valid_identity"]

valid_rows = dedup_view.loc[valid]

strong_mask = valid_rows.duplicated(
    subset=CORE_KEY, keep=False
)

strong_rows = valid_rows.loc[strong_mask]

strong_groups = strong_rows.groupby(
    CORE_KEY, dropna=False
)

# 3. Same event/round/players, potentially
# different match numbers
pair_mask = valid_rows.duplicated(
    subset=EVENT_PAIR_KEY, keep=False
)

pair_rows = valid_rows.loc[pair_mask]

pair_summary = pair_rows.groupby(
    EVENT_PAIR_KEY
).agg(
    records=("winner", "size"),
    distinct_match_nums=("match_no", "nunique")
)

# 4. Possible matches under different event IDs
WEAK_KEY = [
    "date", "name", "round",
    "player_1", "player_2"
]

weak_valid = (
    valid
    & dedup_view[WEAK_KEY].notna().all(axis=1)
)

weak_rows = dedup_view.loc[weak_valid]

weak_rows = weak_rows.loc[
    weak_rows.duplicated(WEAK_KEY, keep=False)
]

weak_summary = weak_rows.groupby(
    WEAK_KEY
).agg(
    records=("event", "size"),
    distinct_events=("event", "nunique"),
    distinct_families=("source_family", "nunique")
)

cross_event = weak_summary[
    weak_summary["distinct_events"] > 1
]

cross_family = strong_groups[
    "source_family"
].nunique() if len(strong_rows) else pd.Series(dtype=int)

print("=== DEDUPLICATION DIAGNOSTICS ===")
print("Total records:", len(matches_df))
print("Exact duplicate rows:", exact_mask.sum())
print("Excess exact copies:", exact_excess)
print("Invalid/incomplete strong identities:", (~valid).sum())
print("Strong collision groups:", strong_groups.ngroups)
print("Records in strong groups:", len(strong_rows))
print("Strong groups spanning families:", (cross_family > 1).sum())
print(
    "Same-event pair groups with different match numbers:",
    (pair_summary["distinct_match_nums"] > 1).sum()
)
print(
    "Possible cross-event duplicate groups:",
    len(cross_event)
)


=== DEDUPLICATION DIAGNOSTICS ===
Total records: 855194
Exact duplicate rows: 0
Excess exact copies: 0
Invalid/incomplete strong identities: 4
Strong collision groups: 0
Records in strong groups: 0
Strong groups spanning families: 0
Same-event pair groups with different match numbers: 22
Possible cross-event duplicate groups: 31


In [21]:
from collections import defaultdict

# Include rows with missing match_num.
base_ok = dedup_view[
    ["round", "player_1", "player_2"]
].notna().all(axis=1)

# Rule A: same event, round and players
event_ok = base_ok & dedup_view["event"].notna()
event_rows = dedup_view.loc[event_ok]

event_ids = event_rows.index[
    event_rows.duplicated(EVENT_PAIR_KEY, keep=False)
]

# Rule B: same reference date, tournament name,
# round and players, but different event IDs.
weak_ok = (
    base_ok &
    dedup_view[WEAK_KEY].notna().all(axis=1)
)

weak_rows = dedup_view.loc[weak_ok]

event_variants = (
    weak_rows["event"]
    .fillna("<MISSING_EVENT>")
    .groupby(
        [weak_rows[c] for c in WEAK_KEY],
        sort=False
    )
    .transform("nunique")
)

cross_ids = weak_rows.index[event_variants.gt(1)]

suspect_ids = event_ids.union(cross_ids)

# Union-find: combine overlapping candidate groups
parent = {int(i): int(i) for i in suspect_ids}

def root(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i

def join(ids):
    ids = [int(i) for i in ids]

    for i in ids[1:]:
        a, b = root(ids[0]), root(i)
        parent[max(a, b)] = min(a, b)

for idx in dedup_view.loc[event_ids].groupby(
    EVENT_PAIR_KEY, sort=False
).groups.values():
    join(idx)

for idx in dedup_view.loc[cross_ids].groupby(
    WEAK_KEY, sort=False
).groups.values():
    join(idx)

candidate = dedup_view.loc[suspect_ids].copy()


candidate["candidate_group_id"] = pd.Series(
    [f"D{root(int(i)):07d}" for i in candidate.index],
    index=candidate.index,
    dtype="object"
)


print("Candidate rows:", len(candidate))
print(
    "Distinct candidate groups:",
    candidate["candidate_group_id"].nunique()
)


Candidate rows: 106
Distinct candidate groups: 53


In [22]:
ignored = {"tourney_id", "match_num"}

compare_cols = [
    c for c in RAW_COLUMNS if c not in ignored
]

stat_cols = [
    c for c in RAW_COLUMNS
    if c.startswith(("w_", "l_")) or c == "minutes"
]

rows = []

for gid, g in candidate.groupby(
    "candidate_group_id", sort=True
):
    orig = matches_df.loc[g.index]

    conflicting = []

    for c in compare_cols:
        values = (
            normalize_text(orig[c])
            .str.casefold()
            .dropna()
            .unique()
        )

        if len(values) > 1:
            conflicting.append(c)

    same_event = (
        g["event"].notna().all()
        and g["event"].nunique() == 1
    )

    same_matchno = (
        g["match_no"].notna().all()
        and g["match_no"].nunique() == 1
    )

    has_score = g["score"].notna().any()

    # Number of statistics observed in at least
    # two records of the candidate group
    shared_stats = int(
        (orig[stat_cols].notna().sum() >= 2).sum()
    )

    if conflicting:
        decision = "CONFLICT"

    elif not has_score:
        decision = "UNRESOLVED_NO_SCORE"

    elif same_event or (
        same_matchno and shared_stats >= 3
    ):
        decision = "AUTO_ELIGIBLE"

    else:
        decision = "UNRESOLVED_IDENTITY"

    rows.append({
        "candidate_group_id": gid,
        "records": len(g),
        "decision": decision,
        "conflicting_fields": ", ".join(conflicting),
        "same_event": same_event,
        "same_match_no": same_matchno,
        "shared_stats": shared_stats,
        "source_families": " | ".join(
            sorted(orig["_source_family"].astype(str).unique())
        ),
        "tournament_names": " | ".join(
            sorted(orig["tourney_name"].dropna().astype(str).unique())
        ),
        "seasons": " | ".join(
            sorted(
                orig["_source_file"]
                .str.extract(r"_(\d{4})\.csv$", expand=False)
                .dropna().unique()
            )
        )
    })

profiles = pd.DataFrame(
    rows,
    columns=[
        "candidate_group_id", "records", "decision",
        "conflicting_fields", "same_event",
        "same_match_no", "shared_stats",
        "source_families", "tournament_names", "seasons"
    ]
)

print(profiles["decision"].value_counts())


decision
UNRESOLVED_IDENTITY    31
AUTO_ELIGIBLE          15
CONFLICT                7
Name: count, dtype: int64


In [23]:
AUDIT_DIR = Path("data/created_data/dedup_audit")
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

# Individual source records for all candidates
members = matches_df.loc[candidate.index].copy()

members.insert(
    0, "candidate_group_id",
    candidate["candidate_group_id"]
)

members = members.merge(
    profiles[["candidate_group_id", "decision"]],
    on="candidate_group_id",
    how="left",
    validate="many_to_one"
)

members = members.sort_values(
    ["candidate_group_id", "_source_file", "_source_row"]
)

assert len(members) == len(candidate)

# Records lacking a complete strong identity
incomplete = matches_df.loc[
    ~dedup_view["valid_identity"]
].copy()

profiles.to_csv(
    AUDIT_DIR / "candidate_groups.csv",
    index=False
)

members.to_csv(
    AUDIT_DIR / "candidate_source_rows.csv",
    index=False
)

incomplete.to_csv(
    AUDIT_DIR / "incomplete_identities.csv",
    index=False
)

print("=== AUTOMATED TRIAGE ===")
print(profiles["decision"].value_counts())
print("Candidate source rows:", len(members))
print("Incomplete strong identities:", len(incomplete))
print(
    "Groups spanning source families:",
    profiles["source_families"].str.contains(
        " | ", regex=False
    ).sum()
)

print("\nAffected seasons:")
print(
    members["_source_file"]
    .str.extract(r"_(\d{4})\.csv$", expand=False)
    .value_counts()
    .sort_index()
)

print("\nAudit directory:", AUDIT_DIR)


=== AUTOMATED TRIAGE ===
decision
UNRESOLVED_IDENTITY    31
AUTO_ELIGIBLE          15
CONFLICT                7
Name: count, dtype: int64
Candidate source rows: 106
Incomplete strong identities: 4
Groups spanning source families: 0

Affected seasons:
_source_file
2000    62
2007    22
2010     2
2011     8
2020     2
2024     4
2025     4
2026     2
Name: count, dtype: int64

Audit directory: data\created_data\dedup_audit


In [24]:

# Focus on unresolved cross-event identities
unresolved_ids = set(
    profiles.loc[
        profiles["decision"] == "UNRESOLVED_IDENTITY",
        "candidate_group_id"
    ]
)

unresolved = candidate.loc[
    candidate["candidate_group_id"].isin(unresolved_ids)
]

pair_records = []

for gid, g in unresolved.groupby("candidate_group_id"):

    events = sorted(g["event"].dropna().unique())

    # Require exactly two records and two event IDs
    if len(g) != 2 or len(events) != 2:
        continue

    orig = matches_df.loc[g.index]

    pair_records.append({
        "candidate_group_id": gid,
        "event_a": events[0],
        "event_b": events[1],
        "date": g["date"].iloc[0],
        "name": g["name"].iloc[0],
        "same_winner": orig["winner_id"].nunique() == 1,
        "same_loser": orig["loser_id"].nunique() == 1,
        "scores_known": (
            normalize_text(orig["score"]).notna().all()
        )
    })

event_pairs = pd.DataFrame(pair_records)

if len(event_pairs):
    alias_table = event_pairs.groupby(
        ["event_a", "event_b", "date", "name"]
    ).agg(
        matched_groups=("candidate_group_id", "nunique"),
        matching_winners=("same_winner", "sum"),
        matching_losers=("same_loser", "sum"),
        pairs_with_scores=("scores_known", "sum")
    ).reset_index()

    # Count every record associated with each event
    event_sizes = dedup_view.groupby(
        ["event", "date", "name"]
    ).size().to_dict()

    for side in ("a", "b"):
        alias_table[f"event_{side}_rows"] = [
            event_sizes.get(
                (event, date, name), 0
            )
            for event, date, name in zip(
                alias_table[f"event_{side}"],
                alias_table["date"],
                alias_table["name"]
            )
        ]

    n = alias_table["matched_groups"]

    # Require a complete, substantial mirrored draw
    alias_table["full_mirror"] = (
        (n >= 8)
        & (n == alias_table["event_a_rows"])
        & (n == alias_table["event_b_rows"])
        & (n == alias_table["matching_winners"])
        & (n == alias_table["matching_losers"])
        & (n == alias_table["pairs_with_scores"])
    )

    print(alias_table.to_string(index=False))

else:
    alias_table = pd.DataFrame()
    print("No two-event candidate pairs found.")


                event_a                 event_b     date         name  matched_groups  matching_winners  matching_losers  pairs_with_scores  event_a_rows  event_b_rows  full_mirror
2000-M-FU-ARG-05A-2000A 2000-M-FU-ARG-05A-2000B 20000515 ARGENTINA F5              31                31               31                 31            31            31         True


In [25]:

conflict_ids = profiles.loc[
    profiles["decision"] == "CONFLICT",
    "candidate_group_id"
]

conflict_details = []

IDENTITY_FIELDS = {
    "winner_id", "loser_id", "score",
    "round", "tourney_name",
    "tourney_date", "surface",
    "tourney_level", "best_of"
}

for gid in conflict_ids:

    group_indices = candidate.index[
        candidate["candidate_group_id"] == gid
    ]

    original = matches_df.loc[group_indices]

    fields = profiles.loc[
        profiles["candidate_group_id"] == gid,
        "conflicting_fields"
    ].iloc[0].split(", ")

    for field in fields:

        values = []

        for _, row in original.iterrows():
            values.append(
                f"{row['_source_file']}:"
                f"{int(row['_source_row'])}="
                f"{row[field]}"
            )

        conflict_details.append({
            "candidate_group_id": gid,
            "field": field,
            "category": (
                "IDENTITY_OR_CONTEXT"
                if field in IDENTITY_FIELDS
                else "STATISTIC_OR_METADATA"
            ),
            "source_values": " | ".join(values)
        })

conflict_details = pd.DataFrame(conflict_details)

print("Conflicting groups:", len(conflict_ids))

if len(conflict_details):
    print("\nConflicting fields:")
    print(
        conflict_details.groupby(
            ["category", "field"]
        ).size().sort_values(ascending=False)
    )

    print("\nField-level disagreements:")
    print(
        conflict_details.to_string(index=False)
    )


Conflicting groups: 7

Conflicting fields:
category               field             
IDENTITY_OR_CONTEXT    loser_id              7
                       score                 7
                       winner_id             7
STATISTIC_OR_METADATA  winner_age            7
                       loser_name            7
                       loser_age             7
                       winner_name           7
                       winner_rank           6
                       loser_ht              6
                       winner_rank_points    6
                       loser_rank_points     6
                       winner_ht             6
                       loser_rank            6
                       l_1stIn               5
                       winner_seed           5
                       l_2ndWon              5
                       minutes               5
                       loser_seed            5
                       loser_ioc             5
                      

In [26]:

# Save event-level duplicate evidence
if not alias_table.empty:
    alias_table.to_csv(
        AUDIT_DIR / "event_alias_candidates.csv",
        index=False
    )

# Save complete disagreement evidence
conflict_details.to_csv(
    AUDIT_DIR / "conflict_details.csv",
    index=False
)

# Summarize current resolution status
verified_mirror_groups = 0

if not alias_table.empty:
    verified_mirror_groups = int(
        alias_table.loc[
            alias_table["full_mirror"],
            "matched_groups"
        ].sum()
    )

print("=== SECOND-STAGE INVESTIGATION ===")

print(
    "Complete tournament mirror groups:",
    verified_mirror_groups
)

print(
    "Potentially compatible same-event groups:",
    (profiles["decision"] == "AUTO_ELIGIBLE").sum()
)

print(
    "Groups with disagreements:",
    len(conflict_ids)
)

print(
    "Still unresolved cross-event groups:",
    len(unresolved_ids) - verified_mirror_groups
)

print("\nReports saved to:", AUDIT_DIR)


=== SECOND-STAGE INVESTIGATION ===
Complete tournament mirror groups: 31
Potentially compatible same-event groups: 15
Groups with disagreements: 7
Still unresolved cross-event groups: 0

Reports saved to: data\created_data\dedup_audit


In [27]:

# Canonical identities for confirmed tournament mirrors.
# The lexicographically first event ID is retained,
# unless future external evidence establishes otherwise.

mirror_keys = set()

if not alias_table.empty:
    mirror_keys = set(
        alias_table.loc[
            alias_table["full_mirror"],
            ["event_a", "event_b", "date", "name"]
        ].itertuples(index=False, name=None)
    )

profile_lookup = profiles.set_index("candidate_group_id")

FAMILY_PRIORITY = {
    "atp_matches": 0,
    "atp_matches_qual_chall": 1,
    "atp_matches_futures": 2
}

def fully_agrees(indices, column):
    values = normalize_text(
        matches_df.loc[indices, column]
    ).str.casefold()

    return values.notna().all() and values.nunique() == 1

decisions = []

for gid, g in candidate.groupby("candidate_group_id", sort=True):

    indices = list(g.index)
    orig = matches_df.loc[indices]
    old_status = profile_lookup.at[gid, "decision"]

    action = "QUARANTINE"
    keep_idx = None
    reason = "Identity not sufficiently verified"

    if len(indices) == 2:

        # Winner and loser must match exactly.
        same_players_result = (
            fully_agrees(indices, "winner_id")
            and fully_agrees(indices, "loser_id")
        )

        # Missing scores are allowed, but recorded
        # scores must not contradict each other.
        scores = g["score"].dropna()

        compatible_score = (
            len(scores) >= 1
            and scores.nunique() == 1
        )

        same_result = (
            same_players_result
            and compatible_score
        )

        same_context = all(
            g[col].notna().all() and g[col].nunique() == 1
            for col in ["date", "name", "round"]
        )

        same_event = (
            g["event"].notna().all()
            and g["event"].nunique() == 1
        )

        events = sorted(g["event"].dropna().tolist())

        mirror_key = None
        if len(events) == 2:
            mirror_key = (
                events[0], events[1],
                g["date"].iloc[0], g["name"].iloc[0]
            )

        if (
            old_status == "AUTO_ELIGIBLE"
            and same_result
            and same_context
            and same_event
        ):
            action = "MERGE"
            reason = "Compatible same-event duplicate"

            # Prefer completeness, followed by stable tie-breaks
            completeness = orig[RAW_COLUMNS].notna().sum(axis=1)

            keep_idx = sorted(
                indices,
                key=lambda i: (
                    -int(completeness.loc[i]),
                    FAMILY_PRIORITY.get(
                        orig.at[i, "_source_family"], 99
                    ),
                    str(orig.at[i, "_source_file"]),
                    int(orig.at[i, "_source_row"])
                )
            )[0]

        elif (
            old_status == "UNRESOLVED_IDENTITY"
            and same_result
            and same_context
            and mirror_key in mirror_keys
        ):
            action = "MERGE"
            reason = "Complete compatible tournament mirror"

            # Retain the canonical event's original row
            canonical_event = events[0]
            keep_idx = next(
                i for i in indices
                if g.at[i, "event"] == canonical_event
            )

        elif (
            old_status == "CONFLICT"
            and same_event
            and g["match_no"].notna().all()
            and g["match_no"].nunique() == 2
        ):
            action = "QUARANTINE"
            reason = (
                "Same event, round and players, "
                "but contradictory match outcomes"
            )

    decisions.append({
        "candidate_group_id": gid,
        "action": action,
        "keep_idx": keep_idx,
        "reason": reason
    })


resolution_plan = pd.DataFrame(
    decisions,
    columns=[
        "candidate_group_id",
        "action",
        "keep_idx",
        "reason"
    ]
)


print("=== FINAL RESOLUTION PLAN ===")
print(resolution_plan["action"].value_counts())


=== FINAL RESOLUTION PLAN ===
action
MERGE         46
QUARANTINE     7
Name: count, dtype: int64


In [28]:

def apply_resolution_plan(frame):

    cleaned = frame.copy(deep=True)
    removals = set()
    field_updates = []

    for record in resolution_plan.itertuples(index=False):

        gid = record.candidate_group_id
        group_indices = list(
            candidate.index[
                candidate["candidate_group_id"] == gid
            ]
        )

        if record.action == "MERGE":

            keeper = int(record.keep_idx)

            for donor in group_indices:

                if donor == keeper or donor not in cleaned.index:
                    continue

                # Do not blend competing tournament identities
                for col in RAW_COLUMNS:

                    if col in {"tourney_id", "match_num"}:
                        continue

                    if (
                        pd.isna(cleaned.at[keeper, col])
                        and pd.notna(cleaned.at[donor, col])
                    ):
                        cleaned.at[keeper, col] = cleaned.at[donor, col]

                        field_updates.append({
                            "candidate_group_id": gid,
                            "keeper_index": keeper,
                            "donor_index": donor,
                            "field": col,
                            "value": cleaned.at[donor, col]
                        })

                removals.add(donor)

        elif record.action == "QUARANTINE":
            removals.update(group_indices)

        # KEEP_BOTH_FLAGGED leaves both rows untouched.

    removals = sorted(removals)

    cleaned = cleaned.drop(
        index=[i for i in removals if i in cleaned.index]
    )

    
    updates_df = pd.DataFrame(
        field_updates,
        columns=[
            "candidate_group_id",
            "keeper_index",
            "donor_index",
            "field",
            "value"
        ]
    )

    return cleaned, removals, updates_df



# Apply the reconciliation
deduped_df, removed_indices, merge_updates = (
    apply_resolution_plan(matches_df)
)

# Applying the same plan again must have no effect
second_pass, _, second_updates = (
    apply_resolution_plan(deduped_df)
)

pd.testing.assert_frame_equal(
    deduped_df, second_pass
)

assert second_updates.empty
assert len(deduped_df) == len(matches_df) - len(removed_indices)

# Verify one survivor per merged group, zero per quarantine
for record in resolution_plan.itertuples(index=False):

    original_indices = set(
        candidate.index[
            candidate["candidate_group_id"]
            == record.candidate_group_id
        ]
    )

    survivors = original_indices.intersection(deduped_df.index)

    expected = {
        "MERGE": 1,
        "QUARANTINE": 0,
        "KEEP_BOTH_FLAGGED": len(original_indices)
    }[record.action]

    assert len(survivors) == expected, record.candidate_group_id

assert not deduped_df[
    ["_source_file", "_source_row"]
].duplicated().any()

print("=== VALIDATION ===")
print("Original records:", len(matches_df))
print("Removed or quarantined:", len(removed_indices))
print("Cleaned records:", len(deduped_df))
print("Missing fields filled:", len(merge_updates))
print("Idempotence: PASSED")
print("Provenance uniqueness: PASSED")
print("Group survivor counts: PASSED")


=== VALIDATION ===
Original records: 855194
Removed or quarantined: 60
Cleaned records: 855134
Missing fields filled: 0
Idempotence: PASSED
Provenance uniqueness: PASSED
Group survivor counts: PASSED


In [29]:

# Save final group decisions
resolution_plan.to_csv(
    AUDIT_DIR / "resolution_decisions.csv",
    index=False
)

# Record exactly what happened to every candidate source row
row_audit = candidate[["candidate_group_id"]].copy()

row_audit["_source_file"] = matches_df.loc[
    row_audit.index, "_source_file"
]

row_audit["_source_row"] = matches_df.loc[
    row_audit.index, "_source_row"
]

row_audit["action"] = row_audit[
    "candidate_group_id"
].map(
    resolution_plan.set_index("candidate_group_id")["action"]
)

row_audit["retained"] = row_audit.index.isin(deduped_df.index)

row_audit.to_csv(
    AUDIT_DIR / "row_decisions.csv",
    index_label="original_dataframe_index"
)

# Preserve all quarantined records separately
quarantine_ids = resolution_plan.loc[
    resolution_plan["action"] == "QUARANTINE",
    "candidate_group_id"
]

quarantine_indices = candidate.index[
    candidate["candidate_group_id"].isin(quarantine_ids)
]

matches_df.loc[quarantine_indices].to_csv(
    AUDIT_DIR / "quarantined_records.csv",
    index=False
)

# Record individual missing-value reconciliations
merge_updates.to_csv(
    AUDIT_DIR / "field_reconciliations.csv",
    index=False
)

# All affected seasons and tournaments
affected_rows = matches_df.loc[candidate.index].copy()
affected_rows["candidate_group_id"] = candidate[
    "candidate_group_id"
]

affected_rows = affected_rows.merge(
    resolution_plan[["candidate_group_id", "action"]],
    on="candidate_group_id",
    how="left"
)

affected_summary = (
    affected_rows.groupby(
        ["tourney_date", "tourney_name", "action"],
        dropna=False
    )
    .size()
    .reset_index(name="source_records")
)

affected_summary.to_csv(
    AUDIT_DIR / "affected_tournaments.csv",
    index=False
)

# Final summary
print("=== DEDUPLICATION COMPLETE ===")
print(resolution_plan["action"].value_counts())
print("Original:", len(matches_df))
print("Final:", len(deduped_df))
print("Removed/quarantined:", len(removed_indices))
print("Unverified identities retained:", len(incomplete))

# Replace only after all validation checks have passed
matches_df = deduped_df.copy()

print("\nAudit saved to:", AUDIT_DIR)
print("Ready to execute original Cell 5.")


=== DEDUPLICATION COMPLETE ===
action
MERGE         46
QUARANTINE     7
Name: count, dtype: int64
Original: 855194
Final: 855134
Removed/quarantined: 60
Unverified identities retained: 4

Audit saved to: data\created_data\dedup_audit
Ready to execute original Cell 5.


In [30]:

# Final audit of records not automatically reconciled
review_actions = resolution_plan[
    resolution_plan["action"].isin(
        ["QUARANTINE", "KEEP_BOTH_FLAGGED"]
    )
]

review = members.merge(
    review_actions[
        ["candidate_group_id", "action", "reason"]
    ],
    on="candidate_group_id",
    how="inner",
    validate="many_to_one"
)

cols = [
    "candidate_group_id", "action",
    "_source_file", "_source_row",
    "tourney_id", "tourney_name",
    "round", "match_num",
    "winner_id", "loser_id", "score"
]

print("=== UNRESOLVED RECORD AUDIT ===")

display(
    review[cols].sort_values(
        ["candidate_group_id", "_source_row"]
    )
)

review.to_csv(
    AUDIT_DIR / "final_unresolved_review.csv",
    index=False
)


expected_review = candidate[
    candidate["candidate_group_id"].isin(
        review_actions["candidate_group_id"]
    )
].shape[0]

assert len(review) == expected_review

assert len(matches_df) == (
    sum(n for _, n in loaded_sources)
    - len(removed_indices)
)


print("Final unresolved audit saved.")


=== UNRESOLVED RECORD AUDIT ===


,candidate_group_id,action,_source_file,_source_row,tourney_id,tourney_name,round,match_num,winner_id,loser_id,score
0,D0337883,QUARANTINE,atp_matches_qual_chall_2007.csv,3462,2007-3666,Kuala Lumpur CH,R32,2,103921,104583,6-3 3-6 6-3
1,D0337883,QUARANTINE,atp_matches_qual_chall_2007.csv,3474,2007-3666,Kuala Lumpur CH,R32,14,104583,103921,7-6(6) 3-6 7-5
2,D0337945,QUARANTINE,atp_matches_qual_chall_2007.csv,3524,2007-3668,Burnie2 CH,R32,2,110685,103977,6-1 6-2
3,D0337945,QUARANTINE,atp_matches_qual_chall_2007.csv,3536,2007-3668,Burnie2 CH,R32,14,103977,110685,6-1 3-6 6-3
4,D0414976,QUARANTINE,atp_matches_qual_chall_2010.csv,3369,2010-468,Estoril,Q3,726,103252,104665,6-3 6-4
5,D0414976,QUARANTINE,atp_matches_qual_chall_2010.csv,3370,2010-468,Estoril,Q3,727,104665,103252,7-6(1) 6-2
6,D0438842,QUARANTINE,atp_matches_qual_chall_2011.csv,988,2011-315,Newport,Q2,710,104372,104330,3-6 7-5 6-3
7,D0438842,QUARANTINE,atp_matches_qual_chall_2011.csv,989,2011-315,Newport,Q2,711,104330,104372,2-6 6-1 6-1
8,D0438844,QUARANTINE,atp_matches_qual_chall_2011.csv,990,2011-315,Newport,Q2,712,104171,104398,6-3 6-2
9,D0438844,QUARANTINE,atp_matches_qual_chall_2011.csv,991,2011-315,Newport,Q2,713,104398,104171,6-1 6-4


Final unresolved audit saved.


In [31]:

# Identify invalid self-matches in the deduplicated data
remaining = dedup_view.loc[matches_df.index]

self_mask = (
    remaining["player_1"]
    .eq(remaining["player_2"])
    .fillna(False)
)

self_match_indices = remaining.index[self_mask]

invalid_records = matches_df.loc[
    self_match_indices
].copy()

invalid_records["_quarantine_reason"] = (
    "Winner and loser have the same player ID"
)

# Preserve invalid records in the audit
invalid_records.to_csv(
    AUDIT_DIR / "invalid_self_matches.csv",
    index=False
)

# Remove them from the modeling dataset
matches_df = matches_df.drop(
    index=self_match_indices
).copy()

# Basic checks
assert matches_df.index.is_unique

assert (
    len(matches_df) + len(invalid_records)
    == len(deduped_df)
)

print("Invalid self-matches quarantined:", len(invalid_records))
print("Records remaining:", len(matches_df))


Invalid self-matches quarantined: 4
Records remaining: 855130


In [32]:
DF_DIR = Path("data/created_data")

matches_df.to_csv(DF_DIR / "matches_1991_2026_combined.csv", index = False)

In [33]:
import pandas as pd
from pathlib import Path

DF_DIR = Path("data/created_data")

matches_df = pd.read_csv(DF_DIR / "matches_1991_2026_combined.csv")

# **Analysing the Data**

In [34]:
import pandas as pd
from pathlib import Path

DF_DIR = Path("data/created_data")

matches_df = pd.read_csv(DF_DIR / "matches_1991_2026_combined.csv")

In [35]:
unique_players = pd.unique(pd.concat([matches_df["winner_id"], matches_df["loser_id"]])).size

unique_tournaments = matches_df["tourney_id"].nunique()

print(f"Unique players: {unique_players}")
print(f"Unique tournaments: {unique_tournaments}")

Unique players: 25232
Unique tournaments: 27179


In [36]:
pd.set_option("display.max_columns", None)
matches_df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,winner_id,winner_seed,winner_entry,winner_name,winner_hand,winner_ht,winner_ioc,winner_age,loser_id,loser_seed,loser_entry,loser_name,loser_hand,loser_ht,loser_ioc,loser_age,score,best_of,round,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,_source_file,_source_family,_source_row
0,1991-339,Adelaide,Hard,32,A,19901231,1,101723,NaN,NaN,Magnus Larsson,R,193.0,SWE,20.7,101414,1.0,NaN,Boris Becker,R,190.0,GER,23.1,6-4 3-6 7-6(2),3,R32,130.0,6.0,2.0,96.0,55.0,39.0,25.0,15.0,2.0,4.0,8.0,3.0,95.0,62.0,44.0,23.0,16.0,6.0,8.0,56.0,NaN,2.0,NaN,atp_matches_1991.csv,atp_matches,2
1,1991-339,Adelaide,Hard,32,A,19901231,2,100946,NaN,Q,Slobodan Zivojinovic,R,198.0,YUG,27.4,101256,NaN,NaN,Mark Kratzmann,L,178.0,AUS,24.6,6-3 3-6 7-6(6),3,R32,119.0,19.0,4.0,101.0,56.0,45.0,25.0,15.0,9.0,10.0,8.0,2.0,84.0,41.0,35.0,27.0,15.0,1.0,2.0,304.0,NaN,75.0,NaN,atp_matches_1991.csv,atp_matches,3
2,1991-339,Adelaide,Hard,32,A,19901231,3,101234,NaN,NaN,Patrik Kuhnen,R,190.0,GER,24.8,101421,NaN,NaN,Veli Paloheimo,R,183.0,FIN,23.0,6-0 6-4,3,R32,71.0,6.0,1.0,54.0,31.0,24.0,13.0,8.0,1.0,1.0,2.0,2.0,60.0,37.0,22.0,6.0,8.0,4.0,8.0,82.0,NaN,69.0,NaN,atp_matches_1991.csv,atp_matches,4
3,1991-339,Adelaide,Hard,32,A,19901231,4,101889,8.0,NaN,Todd Woodbridge,R,178.0,AUS,19.7,101703,NaN,NaN,Guillaume Raoux,R,180.0,FRA,20.8,7-6(2) 6-1,3,R32,85.0,2.0,0.0,60.0,40.0,30.0,14.0,9.0,3.0,3.0,3.0,3.0,74.0,45.0,30.0,11.0,10.0,5.0,8.0,50.0,NaN,84.0,NaN,atp_matches_1991.csv,atp_matches,5
4,1991-339,Adelaide,Hard,32,A,19901231,5,101274,NaN,NaN,Udo Riglewski,R,185.0,GER,24.4,101843,4.0,NaN,Sergi Bruguera,R,188.0,ESP,19.9,7-5 6-3,3,R32,90.0,4.0,2.0,72.0,40.0,33.0,14.0,10.0,7.0,8.0,2.0,2.0,77.0,41.0,28.0,15.0,11.0,4.0,8.0,88.0,NaN,28.0,NaN,atp_matches_1991.csv,atp_matches,6


In [37]:
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 855130 entries, 0 to 855129
Data columns (total 52 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   tourney_id          855130 non-null  str    
 1   tourney_name        855130 non-null  str    
 2   surface             855077 non-null  str    
 3   draw_size           855130 non-null  int64  
 4   tourney_level       855130 non-null  str    
 5   tourney_date        855130 non-null  int64  
 6   match_num           855130 non-null  int64  
 7   winner_id           855130 non-null  int64  
 8   winner_seed         373612 non-null  float64
 9   winner_entry        153908 non-null  str    
 10  winner_name         855130 non-null  str    
 11  winner_hand         855128 non-null  str    
 12  winner_ht           596288 non-null  float64
 13  winner_ioc          855130 non-null  str    
 14  winner_age          849706 non-null  float64
 15  loser_id            855130 non-null  int64  


around 30.4% of the data has the required columns to calculate 'p'.

In [38]:
p_columns = [
    "w_svpt", "w_1stWon", "w_2ndWon",
    "l_svpt", "l_1stWon", "l_2ndWon"
]

complete_p = matches_df[p_columns].notna().all(axis=1)

valid_p = (
    complete_p
    & matches_df["w_svpt"].gt(0)
    & matches_df["l_svpt"].gt(0)
)

for prefix in ["w", "l"]:
    valid_p &= (
        matches_df[f"{prefix}_1stWon"].ge(0)
        & matches_df[f"{prefix}_2ndWon"].ge(0)
        & (
            matches_df[f"{prefix}_1stWon"]
            + matches_df[f"{prefix}_2ndWon"]
            <= matches_df[f"{prefix}_svpt"]
        )
    )

print(f"Total matches: {len(matches_df):,}")
print(f"Complete serve statistics: {complete_p.sum():,}")
print(f"Valid serve probability targets: {valid_p.sum():,}")
print(f"Usable percentage: {valid_p.mean():.2%}")

Total matches: 855,130
Complete serve statistics: 259,664
Valid serve probability targets: 259,407
Usable percentage: 30.34%


In [39]:
pd.set_option("display.max_columns", None)
matches_df.describe()

,draw_size,tourney_date,match_num,winner_id,winner_seed,winner_ht,winner_age,loser_id,loser_seed,loser_ht,loser_age,best_of,minutes,w_ace,w_df,w_svpt,w_1stIn,w_1stWon,w_2ndWon,w_SvGms,w_bpSaved,w_bpFaced,l_ace,l_df,l_svpt,l_1stIn,l_1stWon,l_2ndWon,l_SvGms,l_bpSaved,l_bpFaced,winner_rank,winner_rank_points,loser_rank,loser_rank_points,_source_row
count,855130.000000,8.551300e+05,855130.000000,855130.000000,373612.000000,596288.000000,849706.000000,855130.000000,193839.000000,494185.000000,844190.000000,855130.000000,252712.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259662.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259664.000000,259662.000000,259664.000000,259664.000000,811483.000000,801414.00000,747603.000000,738499.000000,855130.000000
mean,37.036479,2.010063e+07,93.183510,120873.916541,4.821349,184.240318,23.915680,123402.119374,5.558345,184.127224,23.510127,3.057851,101.608867,5.398696,2.607624,72.339154,44.681311,33.154781,15.137589,11.448448,3.389592,5.019548,3.983336,3.367201,74.964697,45.105205,29.190431,13.422057,11.273914,4.598651,8.518616,485.549197,305.69438,603.758123,214.947798,5897.450943
std,20.157322,9.896095e+04,150.376197,35199.096216,3.757418,8.249561,3.864565,36607.568548,4.100817,8.476273,4.061708,0.335194,38.898563,4.639196,2.243456,25.893723,16.939310,11.877943,6.261933,3.597032,2.962940,3.894782,4.054882,2.488441,25.527723,16.974505,12.732865,6.519109,3.600869,3.100302,3.810866,375.337527,824.56271,461.239619,507.007203,4748.389580
min,2.000000,1.990123e+07,1.000000,100113.000000,1.000000,3.000000,14.000000,100063.000000,1.000000,3.000000,14.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-6.000000,0.000000,1.000000,1.00000,1.000000,1.000000,2.000000
25%,32.000000,2.002062e+07,11.000000,103385.000000,2.000000,180.000000,21.000000,103554.000000,3.000000,180.000000,20.400000,3.000000,75.000000,2.000000,1.000000,53.000000,33.000000,25.000000,11.000000,9.000000,1.000000,2.000000,1.000000,2.000000,57.000000,33.000000,20.000000,9.000000,9.000000,2.000000,6.000000,197.000000,28.00000,224.000000,12.000000,1991.000000
50%,32.000000,2.011052e+07,23.000000,104963.000000,4.000000,185.000000,23.500000,105318.000000,5.000000,185.000000,23.000000,3.000000,95.000000,4.000000,2.000000,68.000000,42.000000,31.000000,14.000000,11.000000,3.000000,4.000000,3.000000,3.000000,70.000000,42.000000,27.000000,13.000000,10.000000,4.000000,8.000000,413.000000,84.00000,508.000000,55.000000,4593.000000
75%,32.000000,2.018070e+07,110.000000,111467.000000,7.000000,188.000000,26.300000,122621.000000,7.000000,188.000000,26.000000,3.000000,123.000000,7.000000,4.000000,88.000000,54.000000,40.000000,19.000000,14.000000,5.000000,7.000000,6.000000,5.000000,90.000000,55.000000,36.000000,17.000000,14.000000,6.000000,11.000000,676.000000,244.00000,888.000000,208.000000,9046.000000
max,128.000000,2.026060e+07,1701.000000,214625.000000,35.000000,211.000000,63.400000,214629.000000,38.000000,211.000000,72.100000,5.000000,4756.000000,113.000000,34.000000,491.000000,361.000000,292.000000,82.000000,90.000000,25.000000,34.000000,103.000000,51.000000,489.000000,328.000000,284.000000,101.000000,91.000000,28.000000,38.000000,2260.000000,16950.00000,2267.000000,16950.000000,21292.000000


# **Fixing the Data**

In [40]:
import pandas as pd

DF_DIR = Path("data/created_data")

matches_df = pd.read_csv(DF_DIR / "matches_1991_2026_combined.csv")

In [41]:
matches_df_original_copy = matches_df.copy()

## *Fixing the Datatime Column*

In [42]:
matches_df["tourney_date"] = pd.to_datetime(matches_df["tourney_date"].astype(str), format="%Y%m%d")

In [43]:
matches_df['tourney_date'].head()

0   1990-12-31
1   1990-12-31
2   1990-12-31
3   1990-12-31
4   1990-12-31
Name: tourney_date, dtype: datetime64[us]

## *Arranging in Chronological Order*

In [44]:
matches_df['round'].unique()

<StringArray>
[ 'R32',  'R16',   'QF',   'SF',    'F', 'R128',  'R64',   'RR',   'BR',
   'Q1',   'Q2',   'Q3',   'ER',   'Q4']
Length: 14, dtype: str

In [45]:
ROUND_ORDER = {
    "Q1": 1,
    "Q2": 2,
    "Q3": 3,
    "Q4": 4,
    "R128": 6,
    "R64": 7,
    "R32": 8,
    "R16": 9,
    "RR": 10,
    "QF": 11,
    "SF": 12,
    "BR": 13,
    "F": 14,
    "ER": 15
}

matches_df["round_rank"] = (matches_df["round"].map(ROUND_ORDER))

In [46]:
#this check will catch new round codes
matches_df[matches_df["round_rank"].isna()]["round"].unique()

<StringArray>
[]
Length: 0, dtype: str

In [47]:
#verifying if round ranking looks correct

matches_df[['round', 'round_rank']].drop_duplicates().sort_values(by= 'round_rank')

,round,round_rank
182955,Q1,1
183019,Q2,2
183051,Q3,3
396645,Q4,4
124,R128,6
188,R64,7
0,R32,8
16,R16,9
251,RR,10
24,QF,11


In [48]:
#sorting the values

matches_df = matches_df.sort_values(by= ["tourney_date", "tourney_name", "round_rank"]).reset_index(drop= True)

In [49]:
#verification-1

matches_df[['tourney_date', 'tourney_name', 'round']].head(10)

,tourney_date,tourney_name,round
0,1990-12-29,India Masters 1,R32
1,1990-12-29,India Masters 1,R32
2,1990-12-29,India Masters 1,R32
3,1990-12-29,India Masters 1,R32
4,1990-12-29,India Masters 1,R32
5,1990-12-29,India Masters 1,R32
6,1990-12-29,India Masters 1,R32
7,1990-12-29,India Masters 1,R32
8,1990-12-29,India Masters 1,R32
9,1990-12-29,India Masters 1,R32


In [50]:
#verification-2

matches_df[['tourney_date', 'tourney_name', 'round']].tail(10)

,tourney_date,tourney_name,round
855120,2026-06-01,Tyler CH,R16
855121,2026-06-01,Tyler CH,R16
855122,2026-06-01,Tyler CH,R16
855123,2026-06-01,Tyler CH,QF
855124,2026-06-01,Tyler CH,QF
855125,2026-06-01,Tyler CH,QF
855126,2026-06-01,Tyler CH,QF
855127,2026-06-01,Tyler CH,SF
855128,2026-06-01,Tyler CH,SF
855129,2026-06-01,Tyler CH,F


## *Converting to PlayerA/PlayerB Format*

In [51]:
import numpy as np
import pandas as pd

np.random.seed(42)

random_num = np.random.rand(len(matches_df))

winner_is_A = random_num < 0.5
matches_df['playerA_wins'] = winner_is_A.astype(int) #1 if A wins, 0 if A loses

In [52]:
player_attributes = [
    "id",
    "seed",
    "entry",
    "name",
    "hand",
    "ht",
    "ioc",
    "age",
    "rank",
    "rank_points"
]

for attr in player_attributes:
    winner_col = f"winner_{attr}"
    loser_col = f"loser_{attr}"

    matches_df[f"playerA_{attr}"] = np.where(winner_is_A, matches_df[winner_col], matches_df[loser_col])
    matches_df[f"playerB_{attr}"] = np.where(winner_is_A, matches_df[loser_col], matches_df[winner_col])

In [53]:
stat_attributes = [
    "ace",
    "df",
    "svpt",
    "1stIn",
    "1stWon",
    "2ndWon",
    "SvGms",
    "bpSaved",
    "bpFaced"
]

for attr in stat_attributes:
    winner_col = f"w_{attr}"
    loser_col = f"l_{attr}"

    matches_df[f"playerA_{attr}"] = np.where(winner_is_A, matches_df[winner_col], matches_df[loser_col])
    matches_df[f"playerB_{attr}"] = np.where(winner_is_A, matches_df[loser_col], matches_df[winner_col])

In [54]:
#dropping old columns

winner_loser_cols = [
    col for col in matches_df.columns
    if col.startswith("winner_")
    or col.startswith("loser_")
    or col.startswith("w_")
    or col.startswith("l_")
]

matches_df = matches_df.drop(columns= winner_loser_cols)

In [55]:
matches_df['winner_is_A'] = winner_is_A

In [56]:
#verification-1
matches_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 855130 entries, 0 to 855129
Data columns (total 55 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   tourney_id           855130 non-null  str           
 1   tourney_name         855130 non-null  str           
 2   surface              855077 non-null  str           
 3   draw_size            855130 non-null  int64         
 4   tourney_level        855130 non-null  str           
 5   tourney_date         855130 non-null  datetime64[us]
 6   match_num            855130 non-null  int64         
 7   score                855050 non-null  str           
 8   best_of              855130 non-null  int64         
 9   round                855130 non-null  str           
 10  minutes              252712 non-null  float64       
 11  _source_file         855130 non-null  str           
 12  _source_family       855130 non-null  str           
 13  _source_row          8551

In [57]:
#verification-2
matches_df["playerA_wins"].mean()

np.float64(0.49958953609392726)

In [58]:
#verification-3
matches_df[["playerA_name", "playerB_name", "playerA_wins", "winner_is_A"]].head()

,playerA_name,playerB_name,playerA_wins,winner_is_A
0,Vladimir Gabrichidze,Emilio Marturano,1,True
1,Goran Mihajlovic,Danilo Pila,0,False
2,York Heidenreich,Mark Joachim,0,False
3,Eno Polo,Eugenio Rossi,0,False
4,Srinivasan Vasudevan,Jake Martyn,1,True


In [59]:
matches_df.drop(columns= "winner_is_A", inplace= True)

## *Reseting Index*

In [60]:
matches_df = matches_df.reset_index(drop=True)
matches_df["match_id"] = matches_df.index

In [61]:
matches_df.head()

,tourney_id,tourney_name,surface,draw_size,tourney_level,tourney_date,match_num,score,best_of,round,minutes,_source_file,_source_family,_source_row,round_rank,playerA_wins,playerA_id,playerB_id,playerA_seed,playerB_seed,playerA_entry,playerB_entry,playerA_name,playerB_name,playerA_hand,playerB_hand,playerA_ht,playerB_ht,playerA_ioc,playerB_ioc,playerA_age,playerB_age,playerA_rank,playerB_rank,playerA_rank_points,playerB_rank_points,playerA_ace,playerB_ace,playerA_df,playerB_df,playerA_svpt,playerB_svpt,playerA_1stIn,playerB_1stIn,playerA_1stWon,playerB_1stWon,playerA_2ndWon,playerB_2ndWon,playerA_SvGms,playerB_SvGms,playerA_bpSaved,playerB_bpSaved,playerA_bpFaced,playerB_bpFaced,match_id
0,1991-M-SA-IND-02A-1990a,India Masters 1,Hard,32,S,1990-12-29,1,6-2 7-5,3,R32,NaN,atp_matches_futures_1991.csv,atp_matches_futures,2,8,1,101484,106937,1.0,NaN,NaN,NaN,Vladimir Gabrichidze,Emilio Marturano,R,R,175.0,NaN,GEO,ITA,22.6,23.5,266.0,805.0,82.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,1991-M-SA-IND-02A-1990a,India Masters 1,Hard,32,S,1990-12-29,2,6-3 6-2,3,R32,NaN,atp_matches_futures_1991.csv,atp_matches_futures,3,8,0,107116,106819,NaN,NaN,NaN,Q,Goran Mihajlovic,Danilo Pila,R,R,NaN,NaN,YUG,PHI,19.6,30.2,692.0,941.0,8.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
2,1991-M-SA-IND-02A-1990a,India Masters 1,Hard,32,S,1990-12-29,3,6-1 6-3,3,R32,NaN,atp_matches_futures_1991.csv,atp_matches_futures,4,8,0,108562,102216,NaN,NaN,NaN,LL,York Heidenreich,Mark Joachim,R,R,NaN,NaN,GER,GER,NaN,17.6,862.0,862.0,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,1991-M-SA-IND-02A-1990a,India Masters 1,Hard,32,S,1990-12-29,4,6-4 6-4,3,R32,NaN,atp_matches_futures_1991.csv,atp_matches_futures,5,8,0,106941,101663,8.0,NaN,NaN,NaN,Eno Polo,Eugenio Rossi,R,R,NaN,NaN,KEN,ITA,23.5,21.1,588.0,703.0,14.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
4,1991-M-SA-IND-02A-1990a,India Masters 1,Hard,32,S,1990-12-29,5,6-3 7-6,3,R32,NaN,atp_matches_futures_1991.csv,atp_matches_futures,6,8,1,100794,101480,4.0,NaN,NaN,NaN,Srinivasan Vasudevan,Jake Martyn,R,R,190.0,NaN,IND,GBR,28.9,22.6,420.0,862.0,33.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4


In [62]:
matches_df.to_csv(DF_DIR / "matches_1991_to_2026_cleaned.csv", index = False)

# **Adding Features**

In [63]:
from pathlib import Path
import pandas as pd

DF_DIR = Path("data/created_data")
matches_df = pd.read_csv(DF_DIR / "matches_1991_to_2026_cleaned.csv", parse_dates=["tourney_date"])

In [64]:
matches_df_fixed_copy = matches_df.copy()

## *Matchup Features*

In [65]:
#data test-1
matches_df['playerA_hand'].value_counts(dropna= False)

playerA_hand
R      657785
U      103163
L       93856
A         315
NaN        11
Name: count, dtype: int64

In [66]:
#data test-2
matches_df['playerB_hand'].value_counts(dropna = False)

playerB_hand
R      656925
U      103408
L       94472
A         314
NaN        11
Name: count, dtype: int64

In [67]:
#data fix-1
matches_df['playerA_hand'] = matches_df['playerA_hand'].fillna(value= 'U')
matches_df['playerB_hand'] = matches_df['playerB_hand'].fillna(value= 'U')

matches_df['playerA_hand'].value_counts(dropna= False)

playerA_hand
R    657785
U    103174
L     93856
A       315
Name: count, dtype: int64

In [68]:
#data test-3
matches_df['surface'].isna().sum()

np.int64(53)

In [69]:
# Keep unknown-surface matches through matchup history generation.
# They can contribute to overall H2H, Top-20 and opponent-handedness history.
# We will drop them after this feature group, as the original pipeline did.
matches_df["surface"] = matches_df["surface"].str.capitalize()
matches_df["surface"].isna().sum()

np.int64(53)

In [70]:
#data test-4
matches_df['playerA_wins'].value_counts(dropna=False)

playerA_wins
0    427916
1    427214
Name: count, dtype: int64

In [71]:
# V1.1 matchup features: conservative 10-calendar-day historical cutoff.
# Each match enters the historical dictionaries only when its tournament
# reference date is at least 10 days older than the date being predicted.

from collections import defaultdict, deque
import numpy as np
import pandas as pd

MATCHUP_HISTORY_LAG_DAYS = 10

MATCHUP_FEATURE_COLUMNS = [
    "playerA_h2h_wins", "playerB_h2h_wins", "h2h_total", "h2h_diff",
    "playerA_surface_h2h_wins", "playerB_surface_h2h_wins",
    "surface_h2h_total", "surface_h2h_diff",
    "playerA_last5_h2h_wins", "playerB_last5_h2h_wins",
    "last5_h2h_total", "last5_h2h_diff",
    "playerA_top20_wins", "playerA_top20_losses", "playerA_top20_total",
    "playerA_top20_win_pct", "playerB_top20_wins", "playerB_top20_losses",
    "playerB_top20_total", "playerB_top20_win_pct",
    "playerA_hand_wins", "playerA_hand_losses", "playerA_hand_total",
    "playerA_hand_win_pct", "playerB_hand_wins", "playerB_hand_losses",
    "playerB_hand_total", "playerB_hand_win_pct",
]


def add_matchup_features(frame, history_lag_days=MATCHUP_HISTORY_LAG_DAYS):
    """Add pre-tournament matchup features, without intra-cutoff leakage.

    Input must be sorted by tournament reference date, not actual match date.
    Unknown-surface rows remain in the result and in non-surface history.
    """
    if frame["tourney_date"].isna().any():
        raise ValueError("Matchup features require nonmissing tournament dates.")
    if not frame["tourney_date"].is_monotonic_increasing:
        raise ValueError("Sort by tourney_date before generating matchup features.")

    h2h_record = defaultdict(lambda: [0, 0])
    surface_h2h_record = defaultdict(lambda: [0, 0])
    # Retain all past meetings: equal-date ties near the fifth-most-recent
    # boundary must be evaluated without inventing an internal order.
    last5_h2h_record = defaultdict(list)
    top20_record = defaultdict(lambda: [0, 0])
    hand_record = defaultdict(lambda: defaultdict(lambda: [0, 0]))
    pending = deque()
    features = {name: [] for name in MATCHUP_FEATURE_COLUMNS}
    ambiguous_last5_count = 0

    def commit(past):
        # past = (date, playerA, playerB, A_wins, surface, rankA, rankB,
        #         handA, handB)
        date, a, b, a_wins, surface, rank_a, rank_b, hand_a, hand_b = past
        pair = (min(a, b), max(a, b))
        winner = a if a_wins == 1 else b
        pair_winner_index = 0 if winner == pair[0] else 1

        h2h_record[pair][pair_winner_index] += 1
        last5_h2h_record[pair].append((date, int(winner == pair[0])))

        if pd.notna(surface):
            surface_h2h_record[(pair, surface)][pair_winner_index] += 1

        # Rank is the opponent's rank at the time of the historical match.
        if pd.notna(rank_b) and rank_b <= 20:
            top20_record[a][0 if a_wins == 1 else 1] += 1
        if pd.notna(rank_a) and rank_a <= 20:
            top20_record[b][0 if a_wins == 0 else 1] += 1

        hand_record[a][hand_b][0 if a_wins == 1 else 1] += 1
        hand_record[b][hand_a][0 if a_wins == 0 else 1] += 1

    def last5_counts(history):
        if not history:
            return 0, 0, 0, False
        chosen = history[-5:]
        count = len(chosen)
        if len(history) > 5:
            boundary_date = chosen[0][0]
            all_at_boundary = [outcome for date, outcome in history
                               if date == boundary_date]
            chosen_at_boundary = sum(date == boundary_date
                                     for date, _ in chosen)
            if (chosen_at_boundary < len(all_at_boundary)
                    and len(set(all_at_boundary)) > 1):
                # Unknown ordering changes which results enter the last five.
                return np.nan, np.nan, count, True
        pair0_wins = sum(outcome for _, outcome in chosen)
        return pair0_wins, count - pair0_wins, count, False

    previous_date = None
    cutoff = None
    for row in frame.itertuples(index=False):
        current_date = row.tourney_date
        if previous_date is None or current_date != previous_date:
            cutoff = current_date - pd.Timedelta(days=history_lag_days)
            previous_date = current_date

        # Commit only results that were already eligible BEFORE this date.
        # Rows with the same reference date cannot affect each other.
        while pending and pending[0][0] <= cutoff:
            commit(pending.popleft())

        a, b = row.playerA_id, row.playerB_id
        pair = (min(a, b), max(a, b))
        a_is_first = (a == pair[0])
        surface = row.surface

        # Overall H2H, oriented to the current A/B assignment.
        p0, p1 = h2h_record[pair]
        a_wins, b_wins = (p0, p1) if a_is_first else (p1, p0)
        features["playerA_h2h_wins"].append(a_wins)
        features["playerB_h2h_wins"].append(b_wins)
        features["h2h_total"].append(a_wins + b_wins)
        features["h2h_diff"].append(a_wins - b_wins)

        # Surface H2H. An unknown surface does not define a surface group.
        if pd.notna(surface):
            s0, s1 = surface_h2h_record[(pair, surface)]
            s_a, s_b = (s0, s1) if a_is_first else (s1, s0)
        else:
            s_a, s_b = 0, 0  # these rows are removed after feature generation
        features["playerA_surface_h2h_wins"].append(s_a)
        features["playerB_surface_h2h_wins"].append(s_b)
        features["surface_h2h_total"].append(s_a + s_b)
        features["surface_h2h_diff"].append(s_a - s_b)

        # Last five meetings; tied reference dates can make the boundary
        # ambiguous, but only when the tied outcomes differ.
        p0, p1, n_last5, ambiguous = last5_counts(last5_h2h_record[pair])
        if ambiguous:
            ambiguous_last5_count += 1
        last_a, last_b = (p0, p1) if a_is_first else (p1, p0)
        features["playerA_last5_h2h_wins"].append(last_a)
        features["playerB_last5_h2h_wins"].append(last_b)
        features["last5_h2h_total"].append(n_last5)
        features["last5_h2h_diff"].append(last_a - last_b)

        for player, prefix in ((a, "playerA"), (b, "playerB")):
            wins, losses = top20_record[player]
            n = wins + losses
            features[f"{prefix}_top20_wins"].append(wins)
            features[f"{prefix}_top20_losses"].append(losses)
            features[f"{prefix}_top20_total"].append(n)
            features[f"{prefix}_top20_win_pct"].append(wins / n if n else 0.5)

        for player, opponent_hand, prefix in (
            (a, row.playerB_hand, "playerA"),
            (b, row.playerA_hand, "playerB"),
        ):
            hand = opponent_hand if pd.notna(opponent_hand) else "U"
            wins, losses = hand_record[player][hand]
            n = wins + losses
            features[f"{prefix}_hand_wins"].append(wins)
            features[f"{prefix}_hand_losses"].append(losses)
            features[f"{prefix}_hand_total"].append(n)
            features[f"{prefix}_hand_win_pct"].append(wins / n if n else 0.5)

        # Queue the current result; DO NOT update any history immediately.
        pending.append((
            current_date, a, b, row.playerA_wins, surface,
            row.playerA_rank, row.playerB_rank,
            row.playerA_hand if pd.notna(row.playerA_hand) else "U",
            row.playerB_hand if pd.notna(row.playerB_hand) else "U",
        ))

    result = frame.copy()
    for name, values in features.items():
        result[name] = values
    print(f"Matchup cutoff: {history_lag_days} calendar days")
    print(f"Last-five H2H ambiguity cases: {ambiguous_last5_count:,}")
    return result


# Include all rows in this historical pass, including missing-surface rows.
matches_df = add_matchup_features(matches_df)

# Preserve V1's surface requirement for the model-ready dataset only.
# The missing-surface rows have already contributed to eligible
# overall/Top-20/handedness H2H histories where appropriate.
missing_surface_removed = int(matches_df["surface"].isna().sum())
matches_df = matches_df.dropna(subset=["surface"]).copy()
print(f"Removed {missing_surface_removed:,} unknown-surface rows after matchup history generation")


Matchup cutoff: 10 calendar days
Last-five H2H ambiguity cases: 94
Removed 53 unknown-surface rows after matchup history generation


In [72]:
# V1.1 matchup chronology tests (small synthetic example; no source CSV required)
# Six same-date H2Hs with mixed outcomes -> ambiguous last-five boundary.
# One has unknown surface but must still count toward overall H2H.

_test_rows = []
for i in range(6):
    _test_rows.append({
        "match_id": i, "tourney_date": pd.Timestamp("2020-01-01"),
        "playerA_id": 1, "playerB_id": 2, "playerA_wins": i % 2,
        "surface": np.nan if i == 0 else "Hard",
        "playerA_rank": 40, "playerB_rank": 10,
        "playerA_hand": "R", "playerB_hand": "L",
    })
for mid, date in [(6, "2020-01-09"), (7, "2020-01-11")]:
    _test_rows.append({
        "match_id": mid, "tourney_date": pd.Timestamp(date),
        "playerA_id": 1, "playerB_id": 2, "playerA_wins": 1,
        "surface": "Hard", "playerA_rank": 40, "playerB_rank": 10,
        "playerA_hand": "R", "playerB_hand": "L",
    })
_test_df = pd.DataFrame(_test_rows)
_test_base = add_matchup_features(_test_df)
_test_row9 = _test_base.loc[_test_base["match_id"] == 6].iloc[0]
_test_row11 = _test_base.loc[_test_base["match_id"] == 7].iloc[0]
assert _test_row9["h2h_total"] == 0        # Jan 1 is only 8 days earlier
assert _test_row11["h2h_total"] == 6       # Jan 1 is exactly 10 days earlier
assert _test_row11["surface_h2h_total"] == 5  # unknown-surface match excluded
assert _test_row11["last5_h2h_total"] == 5
assert pd.isna(_test_row11["last5_h2h_diff"])  # equal-date cutoff ambiguity
assert _test_row11["playerA_top20_total"] == 6
assert _test_row11["playerA_hand_total"] == 6

# Same-date row reordering must not change any generated feature by match ID.
_test_shuffled = (
    _test_df.sample(frac=1, random_state=42)
    .sort_values("tourney_date", kind="mergesort")
)
_test_repeat = add_matchup_features(_test_shuffled)
pd.testing.assert_frame_equal(
    _test_base.set_index("match_id")[MATCHUP_FEATURE_COLUMNS].sort_index(),
    _test_repeat.set_index("match_id")[MATCHUP_FEATURE_COLUMNS].sort_index(),
    check_dtype=False,
)
print("Matchup 10-day cutoff, unknown-surface, last-five tie and order-invariance tests passed.")

Matchup cutoff: 10 calendar days
Last-five H2H ambiguity cases: 1
Matchup cutoff: 10 calendar days
Last-five H2H ambiguity cases: 1
Matchup 10-day cutoff, unknown-surface, last-five tie and order-invariance tests passed.


In [73]:
#test-1
test = matches_df[
    (
        (matches_df["playerA_name"] == "Novak Djokovic")
        & (matches_df["playerB_name"] == "Rafael Nadal")
    )
    |
    (
        (matches_df["playerA_name"] == "Rafael Nadal")
        & (matches_df["playerB_name"] == "Novak Djokovic")
    )
]

test[
    [
        "playerA_name",
        "playerB_name",
        "playerA_h2h_wins",
        "playerB_h2h_wins",
        "h2h_total"
    ]
]

,playerA_name,playerB_name,playerA_h2h_wins,playerB_h2h_wins,h2h_total
301448,Novak Djokovic,Rafael Nadal,0,0,0
319129,Rafael Nadal,Novak Djokovic,1,0,1
320373,Novak Djokovic,Rafael Nadal,0,2,2
323597,Rafael Nadal,Novak Djokovic,2,1,3
325440,Novak Djokovic,Rafael Nadal,1,3,4
328319,Novak Djokovic,Rafael Nadal,1,4,5
331377,Novak Djokovic,Rafael Nadal,1,5,6
340282,Novak Djokovic,Rafael Nadal,2,5,7
345902,Rafael Nadal,Novak Djokovic,6,2,8
349677,Rafael Nadal,Novak Djokovic,6,3,9


In [74]:
#test-2
matches_df[
    matches_df["playerA_name"] == "Roger Federer"
][
    [
        "playerA_name",
        "playerA_top20_wins",
        "playerA_top20_losses",
        "playerA_top20_win_pct"
    ]
].tail(10)

,playerA_name,playerA_top20_wins,playerA_top20_losses,playerA_top20_win_pct
685013,Roger Federer,389,165,0.702166
685021,Roger Federer,389,165,0.702166
699751,Roger Federer,389,166,0.700901
701084,Roger Federer,389,166,0.700901
701112,Roger Federer,389,166,0.700901
701760,Roger Federer,389,167,0.699640
702832,Roger Federer,389,167,0.699640
702892,Roger Federer,389,167,0.699640
702902,Roger Federer,389,167,0.699640
702907,Roger Federer,389,167,0.699640


## *ELO Ratings*

In [75]:
#data check-1
matches_df["tourney_level"].unique()

<StringArray>
['S', 'A', 'G', 'C', 'D', 'M', 'F', '15', '25', 'O']
Length: 10, dtype: str

In [76]:
#data fix-1
matches_df["tourney_date"] = pd.to_datetime(matches_df["tourney_date"])

In [77]:
# V1.1 — Elo parameters (single place to tune later; not fit on test data)
BASE_ELO = 1500.0
BASE_K = 24.0  # shared match-level K before modest multipliers
ELO_HISTORY_LAG_DAYS = 10
ELO_MOMENTUM_TOURNAMENTS = 5
ELO_VOLATILITY_TOURNAMENTS = 10

# Keep V1's blending convention.
SURFACE_WEIGHT = 0.70
OVERALL_WEIGHT = 0.30

# Reduced from the larger V1 multipliers to avoid excessive summed batch shifts.
# Unknown levels receive a neutral multiplier of 1.0.
TOURNAMENT_MULTIPLIER = {
    "G": 1.25,  # Grand Slam
    "M": 1.15,  # Masters 1000
    "A": 1.00,  # ATP Tour
    "C": 0.90,  # Challenger
    "F": 0.80,  # Futures
    "S": 0.80,
    "D": 1.00,
    "O": 1.00,
    "15": 0.80,
    "25": 0.80,
}
BEST_OF_FIVE_MULTIPLIER = 1.10

assert np.isclose(SURFACE_WEIGHT + OVERALL_WEIGHT, 1.0)
assert ELO_HISTORY_LAG_DAYS == MATCHUP_HISTORY_LAG_DAYS
print(f"Batch Elo: K={BASE_K:g}, cutoff={ELO_HISTORY_LAG_DAYS}d, "
      f"momentum={ELO_MOMENTUM_TOURNAMENTS} events, "
      f"volatility={ELO_VOLATILITY_TOURNAMENTS} events")

Batch Elo: K=24, cutoff=10d, momentum=5 events, volatility=10 events


In [78]:
# Feature names remain identical to V1 for downstream notebook compatibility.
ELO_FEATURE_COLUMNS = [
    "playerA_elo", "playerB_elo", "elo_diff",
    "playerA_surface_elo", "playerB_surface_elo", "surface_elo_diff",
    "playerA_blended_elo", "playerB_blended_elo", "blended_elo_diff",
    "elo_ratio", "blended_elo_ratio", "surface_elo_ratio",
    "playerA_elo_momentum", "playerB_elo_momentum",
    "playerA_elo_volatility", "playerB_elo_volatility",
]


In [79]:
from collections import defaultdict, deque
from math import fsum
import numpy as np
import pandas as pd


def expected_score(rating_a, rating_b):
    """Expected score for player A; uses pre-tournament ratings only."""
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))


def match_elo_k(tourney_level, best_of):
    """One shared K for both players: keeps every individual update zero-sum."""
    multiplier = TOURNAMENT_MULTIPLIER.get(str(tourney_level), 1.0)
    best_of_multiplier = BEST_OF_FIVE_MULTIPLIER if best_of == 5 else 1.0
    return BASE_K * multiplier * best_of_multiplier


def add_batch_elo_features(frame, history_lag_days=ELO_HISTORY_LAG_DAYS):
    """Compute date-frozen, tournament-batch Elo with delayed historical updates.

    Input: full cleaned match frame, including unknown-surface matches, with
    unique match_id values and nondecreasing tournament reference dates.
    Output: one row per original match, keyed by match_id. No later dataframe
    ordering or previous-match-in-same-tournament result influences a feature.
    """
    necessary = [
        "match_id", "tourney_date", "tourney_id", "tourney_name",
        "playerA_id", "playerB_id", "playerA_wins", "surface",
        "tourney_level", "best_of",
    ]
    missing = sorted(set(necessary) - set(frame.columns))
    if missing:
        raise KeyError(f"Missing Elo input columns: {missing}")

    source = frame.loc[:, necessary].copy()
    source["tourney_date"] = pd.to_datetime(source["tourney_date"])
    source["surface"] = source["surface"].str.capitalize()
    if source["tourney_date"].isna().any():
        raise ValueError("All Elo input rows require a tournament reference date.")
    if not source["tourney_date"].is_monotonic_increasing:
        raise ValueError("Sort Elo input rows by tourney_date first.")
    if source["match_id"].isna().any() or not source["match_id"].is_unique:
        raise ValueError("Elo input must have unique, non-null match_id values.")
    if not source["playerA_wins"].isin([0, 1]).all():
        raise ValueError("Elo requires binary playerA_wins outcomes.")

    overall = defaultdict(lambda: BASE_ELO)
    # Surface ratings are seeded from the current eligible overall rating.
    surface = {}
    snapshots = defaultdict(list)                 # eligible post-update overall Elo
    last_changes = defaultdict(lambda: deque(maxlen=ELO_MOMENTUM_TOURNAMENTS))
    pending = deque()  # (tournament reference date, overall deltas, surface deltas, players)
    features = {name: [] for name in ("match_id", *ELO_FEATURE_COLUMNS)}
    event_count = 0
    released_date_batches = 0

    def surface_rating(player, court):
        if pd.isna(court):
            return np.nan
        key = (player, court)
        if key not in surface:
            surface[key] = overall[player]
        return surface[key]

    def release_batch(batch):
        _, delta_overall, delta_surface, participants = batch
        # All per-player deltas were computed from the old frozen ratings.
        for player in participants:
            change = delta_overall.get(player, 0.0)
            overall[player] += change
            snapshots[player].append(overall[player])
            last_changes[player].append(change)
        for key, change in delta_surface.items():
            surface[key] += change

    def rolling_metrics(player):
        recent = last_changes[player]
        momentum = fsum(recent) if len(recent) >= ELO_MOMENTUM_TOURNAMENTS else 0.0
        history = snapshots[player]
        volatility = (
            float(np.std(history[-ELO_VOLATILITY_TOURNAMENTS:], ddof=0))
            if len(history) >= ELO_VOLATILITY_TOURNAMENTS else 0.0
        )
        return momentum, volatility

    # All events sharing a reference date see exactly the same Elo state.
    # Their effects are committed together when this date becomes eligible.
    for date, day in source.groupby("tourney_date", sort=True):
        cutoff = date - pd.Timedelta(days=history_lag_days)
        while pending and pending[0][0] <= cutoff:
            release_batch(pending.popleft())
            released_date_batches += 1

        day_overall_contributions = defaultdict(list)
        day_surface_contributions = defaultdict(list)
        day_participants = set()

        # A tournament is a date + event id/name group. A player's rating
        # remains frozen across every round (and all simultaneous events).
        for _, event in day.groupby(["tourney_id", "tourney_name"], sort=False,
                                    dropna=False):
            event_count += 1
            event_overall_contributions = defaultdict(list)
            event_surface_contributions = defaultdict(list)
            event_participants = set()

            for row in event.itertuples(index=False):
                a, b = row.playerA_id, row.playerB_id
                court = row.surface
                ra, rb = overall[a], overall[b]
                sa, sb = surface_rating(a, court), surface_rating(b, court)
                wa = int(row.playerA_wins)

                momentum_a, volatility_a = rolling_metrics(a)
                momentum_b, volatility_b = rolling_metrics(b)
                blended_a = SURFACE_WEIGHT * sa + OVERALL_WEIGHT * ra
                blended_b = SURFACE_WEIGHT * sb + OVERALL_WEIGHT * rb

                values = {
                    "match_id": row.match_id,
                    "playerA_elo": ra,
                    "playerB_elo": rb,
                    "elo_diff": ra - rb,
                    "playerA_surface_elo": sa,
                    "playerB_surface_elo": sb,
                    "surface_elo_diff": sa - sb,
                    "playerA_blended_elo": blended_a,
                    "playerB_blended_elo": blended_b,
                    "blended_elo_diff": blended_a - blended_b,
                    "elo_ratio": ra / rb,
                    "blended_elo_ratio": blended_a / blended_b,
                    "surface_elo_ratio": sa / sb,
                    "playerA_elo_momentum": momentum_a,
                    "playerB_elo_momentum": momentum_b,
                    "playerA_elo_volatility": volatility_a,
                    "playerB_elo_volatility": volatility_b,
                }
                for name, value in values.items():
                    features[name].append(value)

                # Shared match K ensures the two players' overall/surface
                # Elo changes are equal and opposite. No margin dependence.
                k = match_elo_k(row.tourney_level, row.best_of)
                delta_a = k * (wa - expected_score(ra, rb))
                event_overall_contributions[a].append(delta_a)
                event_overall_contributions[b].append(-delta_a)
                if pd.notna(court):
                    delta_surface_a = k * (wa - expected_score(sa, sb))
                    event_surface_contributions[(a, court)].append(delta_surface_a)
                    event_surface_contributions[(b, court)].append(-delta_surface_a)
                event_participants.update((a, b))

            # Aggregate every match for this tournament, before committing.
            for player, deltas in event_overall_contributions.items():
                day_overall_contributions[player].append(fsum(deltas))
            for key, deltas in event_surface_contributions.items():
                day_surface_contributions[key].append(fsum(deltas))
            day_participants.update(event_participants)

        # Do not order two tournaments with identical reference dates.
        # Combining their per-player deltas also gives exactly one rolling
        # history entry per date, rather than an arbitrary event sequence.
        pending.append((
            date,
            {p: fsum(v) for p, v in day_overall_contributions.items()},
            {key: fsum(v) for key, v in day_surface_contributions.items()},
            day_participants,
        ))

    result = pd.DataFrame(features)
    assert len(result) == len(source) and result["match_id"].is_unique
    print(f"Batch Elo: {event_count:,} tournament groups from {len(source):,} matches; "
          f"{released_date_batches:,} reference-date batches released by last row")
    return result


In [80]:
# Compute Elo using the FULL frame retained before the matchup block's
# missing-surface filter; do not lose overall Elo evidence from those matches.
elo_full = add_batch_elo_features(
    matches_df_fixed_copy, history_lag_days=ELO_HISTORY_LAG_DAYS
)


Batch Elo: 27,179 tournament groups from 855,130 matches; 2,730 reference-date batches released by last row


In [81]:
# Attach to the model-ready rows by stable match_id (never by row position).
elo_lookup = elo_full.set_index("match_id")
if not matches_df["match_id"].isin(elo_lookup.index).all():
    raise ValueError("Some modelling rows are missing batch-Elo records.")

matches_df[ELO_FEATURE_COLUMNS] = elo_lookup.loc[
    matches_df["match_id"], ELO_FEATURE_COLUMNS
].to_numpy()

assert matches_df[ELO_FEATURE_COLUMNS].notna().all().all(), (
    "Unexpected missing Elo features in modelling rows"
)
print("Attached", len(ELO_FEATURE_COLUMNS), "Elo features by match_id")


Attached 16 Elo features by match_id


In [82]:
# V1.1 Batch-Elo regression checks (synthetic only; no CSV required)
from pandas.testing import assert_frame_equal


def _elo_row(mid, date, event, a, b, a_wins, court="Hard", level="A"):
    return {
        "match_id": mid, "tourney_date": pd.Timestamp(date),
        "tourney_id": event, "tourney_name": f"Event {event}",
        "playerA_id": a, "playerB_id": b, "playerA_wins": a_wins,
        "surface": court, "tourney_level": level, "best_of": 3,
    }


# (1) Same-tournament rows freeze Elo, even across rounds/matches.
# (2) Jan 8 must not see Jan 1; Jan 11 must see Jan 1 (exactly 10 days).
_smoke = pd.DataFrame([
    _elo_row(0, "2020-01-01", "x", 1, 2, 1),
    _elo_row(1, "2020-01-01", "x_parallel", 1, 3, 1),
    _elo_row(2, "2020-01-08", "y", 1, 4, 0),
    _elo_row(3, "2020-01-11", "z", 1, 5, 1),
])
_before = add_batch_elo_features(_smoke).set_index("match_id")
assert _before.at[0, "playerA_elo"] == BASE_ELO
assert _before.at[1, "playerA_elo"] == BASE_ELO
assert _before.at[2, "playerA_elo"] == BASE_ELO
assert np.isclose(_before.at[3, "playerA_elo"], BASE_ELO + BASE_K)

# (3) Zero-sum pairwise update and unknown-surface overall contribution.
_missing = pd.DataFrame([
    _elo_row(0, "2020-01-01", "m", 1, 2, 1, court=np.nan),
    _elo_row(1, "2020-01-11", "n", 1, 2, 1),
])
_missing_result = add_batch_elo_features(_missing).set_index("match_id")
assert np.isclose(_missing_result.at[1, "playerA_elo"], BASE_ELO + BASE_K / 2)
assert np.isclose(_missing_result.at[1, "playerB_elo"], BASE_ELO - BASE_K / 2)

# (4) Same-date row order and event order must be irrelevant by match_id.
_shuffled = (_smoke.sample(frac=1, random_state=19)
             .sort_values("tourney_date", kind="mergesort"))
_repeat = add_batch_elo_features(_shuffled).set_index("match_id")
assert_frame_equal(
    _before[ELO_FEATURE_COLUMNS].sort_index(),
    _repeat[ELO_FEATURE_COLUMNS].sort_index(),
    check_exact=False, atol=1e-10, rtol=1e-12,
)

# (5) Rolling metrics use five and ten *eligible tournament-date updates*.
_history = pd.DataFrame([
    _elo_row(i, pd.Timestamp("2020-01-01") + pd.Timedelta(days=14 * i),
             f"E{i}", 1, 2, i % 2)
    for i in range(12)
])
_history_result = add_batch_elo_features(_history).set_index("match_id")
assert (_history_result.loc[:4, "playerA_elo_momentum"] == 0).all()
assert np.isclose(
    _history_result.at[5, "playerA_elo_momentum"],
    _history_result.at[5, "playerA_elo"] - BASE_ELO,
)
assert (_history_result.loc[:9, "playerA_elo_volatility"] == 0).all()
assert _history_result.at[10, "playerA_elo_volatility"] > 0
print("Batch Elo: 10-day cutoff, frozen dates, zero-sum, order invariance, "
      "missing-surface and 5/10-history tests passed.")


Batch Elo: 4 tournament groups from 4 matches; 1 reference-date batches released by last row
Batch Elo: 2 tournament groups from 2 matches; 1 reference-date batches released by last row
Batch Elo: 4 tournament groups from 4 matches; 1 reference-date batches released by last row
Batch Elo: 12 tournament groups from 12 matches; 11 reference-date batches released by last row
Batch Elo: 10-day cutoff, frozen dates, zero-sum, order invariance, missing-surface and 5/10-history tests passed.


In [83]:
#test-1
matches_df[
    ["playerA_elo",
     "playerB_elo",
     "playerA_surface_elo",
     "playerB_surface_elo"]
].describe()

,playerA_elo,playerB_elo,playerA_surface_elo,playerB_surface_elo
count,855077.000000,855077.000000,855077.000000,855077.000000
mean,1659.616921,1659.666468,1608.496865,1608.542220
std,199.532709,199.743043,153.668214,153.931704
min,1198.049878,1198.049878,1243.803292,1243.803292
25%,1500.000000,1500.000000,1490.304286,1490.134810
50%,1616.415977,1616.568659,1571.626425,1571.410910
75%,1771.850404,1772.064676,1696.756336,1697.094889
max,2808.007909,2808.007909,2594.709334,2585.474923


In [84]:
#test-2
matches_df[
    [
        "playerA_name",
        "playerB_name",
        "playerA_elo",
        "playerB_elo"
    ]
].head(10)

,playerA_name,playerB_name,playerA_elo,playerB_elo
0,Vladimir Gabrichidze,Emilio Marturano,1500.0,1500.0
1,Goran Mihajlovic,Danilo Pila,1500.0,1500.0
2,York Heidenreich,Mark Joachim,1500.0,1500.0
3,Eno Polo,Eugenio Rossi,1500.0,1500.0
4,Srinivasan Vasudevan,Jake Martyn,1500.0,1500.0
5,Asif Ismail,Mohesh Koregaonkar,1500.0,1500.0
6,Sebastien Hette,Mark Ferreira,1500.0,1500.0
7,Manuel Gasbarri,Nick Adams,1500.0,1500.0
8,Enrico Piperno,Rohit Rajpal,1500.0,1500.0
9,David Kvaratshelia,Massimo Valeri,1500.0,1500.0


In [85]:
#test-3
matches_df[
    matches_df["playerA_name"] == "Roger Federer"
][
    [
        "playerA_name",
        "playerA_elo"
    ]
].tail(20)

,playerA_name,playerA_elo
675483,Roger Federer,2565.856983
675511,Roger Federer,2565.856983
675525,Roger Federer,2565.856983
675532,Roger Federer,2565.856983
677489,Roger Federer,2547.123252
679705,Roger Federer,2554.756999
680387,Roger Federer,2540.130563
680403,Roger Federer,2540.130563
680415,Roger Federer,2540.130563
682712,Roger Federer,2556.838905


In [86]:
#test-4
print("Overall Elo Range:")
print(matches_df["playerA_elo"].min(), matches_df["playerA_elo"].max())

print("\nSurface Elo Range:")
print(matches_df["playerA_surface_elo"].min(), matches_df["playerA_surface_elo"].max())

print("\nBlended Elo Range:")
print(matches_df["playerA_blended_elo"].min(), matches_df["playerA_blended_elo"].max())

Overall Elo Range:
1198.0498783420287 2808.007909073021

Surface Elo Range:
1243.8032919983707 2594.709333721442

Blended Elo Range:
1230.077267901468 2651.3334453502107


In [87]:
#test-5
print("\nMissing Values:")
print(matches_df[
    [
        "playerA_elo",
        "playerA_surface_elo",
        "playerA_blended_elo",
        "playerA_elo_momentum",
        "playerA_elo_volatility",
    ]
].isna().sum())


Missing Values:
playerA_elo               0
playerA_surface_elo       0
playerA_blended_elo       0
playerA_elo_momentum      0
playerA_elo_volatility    0
dtype: int64


In [88]:
#test-6
print("\nExample Features:")
print(
    matches_df[
        [
            "playerA_elo",
            "playerA_surface_elo",
            "playerA_blended_elo",
            "playerA_elo_momentum",
            "playerA_elo_volatility",
        ]
    ].head()
)


Example Features:
   playerA_elo  playerA_surface_elo  playerA_blended_elo  \
0       1500.0               1500.0               1500.0   
1       1500.0               1500.0               1500.0   
2       1500.0               1500.0               1500.0   
3       1500.0               1500.0               1500.0   
4       1500.0               1500.0               1500.0   

   playerA_elo_momentum  playerA_elo_volatility  
0                   0.0                     0.0  
1                   0.0                     0.0  
2                   0.0                     0.0  
3                   0.0                     0.0  
4                   0.0                     0.0  


## *Recent Form*


In [89]:
# V1.1 recent form: cumulative and date-blocked rolling statistics using
# exactly the same 10-calendar-day historical eligibility policy as matchup/Elo.
from collections import defaultdict, deque, Counter
import numpy as np
import pandas as pd

RECENT_HISTORY_LAG_DAYS = 10
RECENT_52W_DAYS = 364
RECENT_WINDOW_CONFIG = {10: (8, 12), 25: (22, 28)}
assert RECENT_HISTORY_LAG_DAYS == MATCHUP_HISTORY_LAG_DAYS == ELO_HISTORY_LAG_DAYS


def _select_recent_window(groups, target, lower, upper):
    """Return (wins, count, pct, is_ambiguous), respecting tied date blocks.

    Groups are chronological (date, wins, matches). No order is inferred
    among matches with the same reference date. Counts may be < lower when
    the player's entire eligible career is shorter than `target`.
    """
    if not groups:
        return 0, 0, 0.5, False

    total = sum(group[2] for group in groups)
    if total < target:
        # Entire known eligible history is unambiguous, even for new players.
        wins = sum(group[1] for group in groups)
        return wins, total, wins / total, False

    # Take the closest feasible suffix consisting of WHOLE date blocks.
    choices = []
    count = 0
    wins = 0
    for _, block_wins, block_count in reversed(groups):
        count += block_count
        wins += block_wins
        if lower <= count <= upper:
            choices.append((abs(count - target), -count, wins, count))
        if count > upper:
            break

    if choices:
        _, _, chosen_wins, chosen_count = min(choices)
        return chosen_wins, chosen_count, chosen_wins / chosen_count, False

    # No admissible whole-block suffix. The exact N-match count is still
    # identifiable when the partially selected boundary block is all W or L.
    count = 0
    wins = 0
    for _, block_wins, block_count in reversed(groups):
        needed = target - count
        if block_count <= needed:
            count += block_count
            wins += block_wins
        else:
            if block_wins == 0:
                return wins, target, wins / target, False
            if block_wins == block_count:
                wins += needed
                return wins, target, wins / target, False
            return np.nan, np.nan, np.nan, True
        if count == target:
            return wins, target, wins / target, False
    raise AssertionError("History ended before target despite total >= target")


RECENT_FEATURE_COLUMNS = []
for prefix in ("playerA", "playerB"):
    for group in ("career", "surface_career"):
        RECENT_FEATURE_COLUMNS.extend([
            f"{prefix}_{group}_wins", f"{prefix}_{group}_losses",
            f"{prefix}_{group}_matches", f"{prefix}_{group}_pct",
        ])
    for group in ("last10", "last25", "surface10", "surface25", "52w"):
        RECENT_FEATURE_COLUMNS.extend([
            f"{prefix}_{group}_wins", f"{prefix}_{group}_losses",
            f"{prefix}_{group}_matches", f"{prefix}_{group}_pct",
        ])


def add_recent_form_features(frame, history_lag_days=RECENT_HISTORY_LAG_DAYS,
                             report=True):
    """Make leakage-controlled recent-form features, keyed by match_id.

    Input includes all matches (also those with unknown surface) and must be
    chronologically sorted by tournament *reference* date, not match date.
    """
    required = ["match_id", "tourney_date", "playerA_id", "playerB_id",
                "playerA_wins", "surface"]
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f"Missing recent-form input columns: {missing}")

    source = frame.loc[:, required].copy()
    source["tourney_date"] = pd.to_datetime(source["tourney_date"])
    source["surface"] = source["surface"].str.capitalize()
    if source["tourney_date"].isna().any():
        raise ValueError("Recent form requires nonmissing tournament reference dates")
    if not source["tourney_date"].is_monotonic_increasing:
        raise ValueError("Sort input by tourney_date before adding recent-form features")
    if source["match_id"].isna().any() or not source["match_id"].is_unique:
        raise ValueError("Recent-form input match_id must be unique and nonmissing")
    if not source["playerA_wins"].isin((0, 1)).all():
        raise ValueError("Recent-form outcomes must be binary")

    career = defaultdict(lambda: [0, 0])          # player -> [wins, matches]
    surface_career = defaultdict(lambda: [0, 0])  # (player, surface)
    recent = defaultdict(deque)                    # last ~28 whole date blocks
    surface_recent = defaultdict(deque)
    recent_count = defaultdict(int)
    surface_recent_count = defaultdict(int)
    rolling_52w = defaultdict(deque)               # (date, wins, matches)
    rolling_52w_count = defaultdict(int)
    rolling_52w_wins = defaultdict(int)
    pending = deque()                              # completed reference-date blocks
    output = {col: [] for col in ("match_id", *RECENT_FEATURE_COLUMNS)}
    ambiguity = Counter()

    def append_recent(key, date, block_wins, block_count, records, totals):
        hist = records[key]
        hist.append((date, block_wins, block_count))
        totals[key] += block_count
        # Keep the latest 28 matches PLUS their full boundary date group.
        while len(hist) > 1 and totals[key] - hist[0][2] >= 28:
            totals[key] -= hist.popleft()[2]

    def release(date, by_player, by_surface):
        for player, (wins, count) in by_player.items():
            career[player][0] += wins
            career[player][1] += count
            append_recent(player, date, wins, count, recent, recent_count)
            rolling_52w[player].append((date, wins, count))
            rolling_52w_wins[player] += wins
            rolling_52w_count[player] += count
        for key, (wins, count) in by_surface.items():
            surface_career[key][0] += wins
            surface_career[key][1] += count
            append_recent(key, date, wins, count,
                          surface_recent, surface_recent_count)

    def get_player_snapshot(player, cutoff):
        wins, count = career[player]
        snapshot = {"career": (wins, count, wins / count if count else 0.5)}
        for name, target in (("last10", 10), ("last25", 25)):
            lo, hi = RECENT_WINDOW_CONFIG[target]
            window_wins, window_count, pct, is_ambiguous = _select_recent_window(
                recent[player], target, lo, hi
            )
            if is_ambiguous:
                ambiguity[name] += 1
            snapshot[name] = (window_wins, window_count, pct)

        # 52 complete weeks END at the historical eligibility cutoff.
        oldest_allowed = cutoff - pd.Timedelta(days=RECENT_52W_DAYS)
        hist = rolling_52w[player]
        while hist and hist[0][0] <= oldest_allowed:
            _, w, n = hist.popleft()
            rolling_52w_wins[player] -= w
            rolling_52w_count[player] -= n
        w, n = rolling_52w_wins[player], rolling_52w_count[player]
        snapshot["52w"] = (w, n, w / n if n else 0.5)
        return snapshot

    def get_surface_snapshot(player, court):
        if pd.isna(court):
            return {name: (np.nan, np.nan, np.nan)
                    for name in ("surface_career", "surface10", "surface25")}
        key = (player, court)
        wins, count = surface_career[key]
        snapshot = {"surface_career": (wins, count,
                                         wins / count if count else 0.5)}
        for name, target in (("surface10", 10), ("surface25", 25)):
            lo, hi = RECENT_WINDOW_CONFIG[target]
            window_wins, window_count, pct, is_ambiguous = _select_recent_window(
                surface_recent[key], target, lo, hi
            )
            if is_ambiguous:
                ambiguity[name] += 1
            snapshot[name] = (window_wins, window_count, pct)
        return snapshot

    for date, day in source.groupby("tourney_date", sort=True):
        cutoff = date - pd.Timedelta(days=history_lag_days)
        while pending and pending[0][0] <= cutoff:
            release(*pending.popleft())

        overall_today = defaultdict(lambda: [0, 0])
        surface_today = defaultdict(lambda: [0, 0])
        player_cache = {}
        surface_cache = {}

        for row in day.itertuples(index=False):
            a, b = row.playerA_id, row.playerB_id
            wa = int(row.playerA_wins)
            court = row.surface
            output["match_id"].append(row.match_id)

            for player, label in ((a, "playerA"), (b, "playerB")):
                if player not in player_cache:
                    player_cache[player] = get_player_snapshot(player, cutoff)
                snapshot = player_cache[player]
                surface_key = (player, court)
                if surface_key not in surface_cache:
                    surface_cache[surface_key] = get_surface_snapshot(player, court)
                snapshot_surface = surface_cache[surface_key]

                for group in ("career", "last10", "last25", "52w"):
                    wins, count, pct = snapshot[group]
                    output[f"{label}_{group}_wins"].append(wins)
                    output[f"{label}_{group}_losses"].append(count - wins)
                    output[f"{label}_{group}_matches"].append(count)
                    output[f"{label}_{group}_pct"].append(pct)
                for group in ("surface_career", "surface10", "surface25"):
                    wins, count, pct = snapshot_surface[group]
                    output[f"{label}_{group}_wins"].append(wins)
                    output[f"{label}_{group}_losses"].append(count - wins)
                    output[f"{label}_{group}_matches"].append(count)
                    output[f"{label}_{group}_pct"].append(pct)

            # Collect completed results without exposing them to same-date rows.
            for player, won in ((a, wa), (b, 1 - wa)):
                overall_today[player][0] += won
                overall_today[player][1] += 1
                if pd.notna(court):
                    surface_today[(player, court)][0] += won
                    surface_today[(player, court)][1] += 1

        pending.append((date, overall_today, surface_today))

    result = pd.DataFrame(output)
    if len(result) != len(source) or not result["match_id"].is_unique:
        raise AssertionError("Recent-form output is not one-to-one by match_id")
    if report:
        print(f"Recent form: {len(source):,} matches, cutoff={history_lag_days} days; "
              f"date-boundary ambiguities: {dict(ambiguity)}")
    return result


In [90]:
# Compute from the full pre-filter history, including unknown-surface rows.
recent_full = add_recent_form_features(
    matches_df_fixed_copy, history_lag_days=RECENT_HISTORY_LAG_DAYS
)
recent_lookup = recent_full.set_index("match_id")
if not matches_df["match_id"].isin(recent_lookup.index).all():
    raise ValueError("Missing recent-form features for one or more modeling rows")

# Attach by match_id rather than dataframe position; preserve all V1 names.
matches_df[RECENT_FEATURE_COLUMNS] = recent_lookup.loc[
    matches_df["match_id"], RECENT_FEATURE_COLUMNS
].to_numpy()
print("Attached", len(RECENT_FEATURE_COLUMNS), "recent-form columns by match_id")


Recent form: 855,130 matches, cutoff=10 days; date-boundary ambiguities: {'last10': 20890, 'surface10': 20009, 'last25': 11823, 'surface25': 10718}
Attached 56 recent-form columns by match_id


In [91]:
# V1.1 recent-form synthetic tests: exact lag, flexible windows,
# missing-surface contribution, 52-week boundaries, and order invariance.
from pandas.testing import assert_frame_equal


def _form_row(mid, date, a, b, wa, court="Hard"):
    return {
        "match_id": mid, "tourney_date": pd.Timestamp(date),
        "playerA_id": a, "playerB_id": b, "playerA_wins": wa,
        "surface": court,
    }

# A's first ten results occur in a single date group, with both outcomes.
_rows = [_form_row(i, "2020-01-01", 1, 100+i, i % 2,
                  np.nan if i == 0 else "Hard") for i in range(10)]
_rows.extend([
    _form_row(10, "2020-01-09", 1, 201, 1),
    _form_row(11, "2020-01-11", 1, 202, 1),
])
_test = add_recent_form_features(pd.DataFrame(_rows), report=False).set_index("match_id")
assert _test.at[10, "playerA_career_matches"] == 0  # 8 days old: not eligible
assert _test.at[11, "playerA_career_matches"] == 10
assert _test.at[11, "playerA_last10_matches"] == 10
assert _test.at[11, "playerA_surface_career_matches"] == 9
assert _test.at[11, "playerA_52w_matches"] == 10

# Whole same-date groups: 8+4, choose 12 rather than an arbitrary 10.
_g = ([_form_row(i, "2020-01-01", 1, 100+i, i % 2) for i in range(4)]
      + [_form_row(i+4, "2020-01-15", 1, 104+i, i % 2) for i in range(8)]
      + [_form_row(12, "2020-02-01", 1, 300, 1)])
_gresult = add_recent_form_features(pd.DataFrame(_g), report=False).set_index("match_id")
assert _gresult.at[12, "playerA_last10_matches"] == 12
assert _gresult.at[12, "playerA_last25_matches"] == 12  # short career

# Newest group of 14 mixed outcomes cannot make any valid 8-12 suffix.
_many = [_form_row(i, "2020-01-01", 1, 500+i, i % 2)
         for i in range(14)]
_many.append(_form_row(14, "2020-01-20", 1, 900, 1))
_mresult = add_recent_form_features(pd.DataFrame(_many), report=False).set_index("match_id")
for col in ("playerA_last10_wins", "playerA_last10_losses",
            "playerA_last10_matches", "playerA_last10_pct"):
    assert pd.isna(_mresult.at[14, col])
assert _mresult.at[14, "playerA_last25_matches"] == 14

# Identical-outcome group allows a deterministic exact 10-match fallback.
_homogeneous = [_form_row(i, "2020-01-01", 1, 500+i, 1)
                for i in range(14)]
_homogeneous.append(_form_row(14, "2020-01-20", 1, 900, 1))
_hres = add_recent_form_features(pd.DataFrame(_homogeneous), report=False).set_index("match_id")
assert _hres.at[14, "playerA_last10_matches"] == 10
assert _hres.at[14, "playerA_last10_wins"] == 10

# 25-match flexible selection: 28 in a whole date group is acceptable.
_25 = [_form_row(i, "2020-01-01", 1, 700+i, i % 2) for i in range(28)]
_25.append(_form_row(28, "2020-01-20", 1, 999, 1))
_25res = add_recent_form_features(pd.DataFrame(_25), report=False).set_index("match_id")
assert _25res.at[28, "playerA_last25_matches"] == 28

# 52 weeks: strict lower bound (T-374, T-10] and full 364-day span.
_wdates = ["2020-01-01", "2020-01-02", "2020-12-30", "2021-01-10"]
_w = [_form_row(i, date, 1, 600+i, 1) for i, date in enumerate(_wdates)]
_w.append(_form_row(4, "2021-01-20", 1, 777, 1))
_wresult = add_recent_form_features(pd.DataFrame(_w), report=False).set_index("match_id")
# The oldest permitted date is Jan 22, 2020 for Jan 20, 2021's cutoff.
assert _wresult.at[4, "playerA_52w_matches"] == 2
assert _wresult.at[4, "playerA_career_matches"] == 4

# Check permutation invariance within dates, including ambiguous windows.
_shuf = pd.DataFrame(_many).sample(frac=1, random_state=31).sort_values(
    "tourney_date", kind="mergesort")
_repeat = add_recent_form_features(_shuf, report=False).set_index("match_id")
assert_frame_equal(
    _mresult[RECENT_FEATURE_COLUMNS].sort_index(),
    _repeat[RECENT_FEATURE_COLUMNS].sort_index(),
    check_exact=False, atol=1e-12, rtol=1e-12,
)
print("Recent-form validation passed: lag, group windows, ambiguity, "
      "unknown surface, 52-week boundary and row-order invariance.")


Recent-form validation passed: lag, group windows, ambiguity, unknown surface, 52-week boundary and row-order invariance.


In [92]:
#test-1
matches_df[matches_df["playerB_name"] == "Roger Federer"][[
                                                    "playerB_name",
                                                    "playerB_career_wins",
                                                    "playerB_career_losses",
                                                    "playerB_career_pct"]].tail(10)

,playerB_name,playerB_career_wins,playerB_career_losses,playerB_career_pct
684943,Roger Federer,1291.0,292.0,0.815540
684983,Roger Federer,1291.0,292.0,0.815540
685003,Roger Federer,1291.0,292.0,0.815540
685018,Roger Federer,1291.0,292.0,0.815540
695953,Roger Federer,1296.0,293.0,0.815607
695961,Roger Federer,1296.0,293.0,0.815607
701028,Roger Federer,1297.0,295.0,0.814698
701126,Roger Federer,1297.0,295.0,0.814698
701770,Roger Federer,1300.0,296.0,0.814536
702872,Roger Federer,1301.0,297.0,0.814143


In [93]:
#test-2
matches_df[["playerA_name", "playerA_last25_matches", "playerA_last25_pct"]].tail(20)

,playerA_name,playerA_last25_matches,playerA_last25_pct
855110,Nicolas Mejia,27.0,0.592593
855111,Edas Butvilas,25.0,0.600000
855112,Fajing Sun,25.0,0.680000
855113,Murphy Cassone,27.0,0.518519
855114,Yuta Shimizu,26.0,0.500000
855115,Andres Martin,25.0,0.360000
855116,Timo Legout,26.0,0.653846
855117,Liam Draxl,26.0,0.576923
855118,Braden Shick,24.0,0.666667
855119,Hayato Matsuoka,24.0,0.583333


In [94]:
#test-3
matches_df[["tourney_date", "playerA_52w_matches", "playerA_52w_pct"]].tail(15)

,tourney_date,playerA_52w_matches,playerA_52w_pct
855115,2026-06-01,60.0,0.466667
855116,2026-06-01,9.0,0.444444
855117,2026-06-01,70.0,0.585714
855118,2026-06-01,65.0,0.692308
855119,2026-06-01,69.0,0.579710
855120,2026-06-01,54.0,0.388889
855121,2026-06-01,75.0,0.560000
855122,2026-06-01,51.0,0.490196
855123,2026-06-01,9.0,0.444444
855124,2026-06-01,70.0,0.585714


In [95]:
#test-4
matches_df["playerA_52w_matches"].max()

np.float64(133.0)

In [96]:
#test-5
matches_df[matches_df["playerA_name"] == "Roger Federer"][["playerA_name", "playerA_52w_matches", "playerA_52w_pct"]].head(10)

,playerA_name,playerA_52w_matches,playerA_52w_pct
100771,Roger Federer,0.0,0.500000
116943,Roger Federer,1.0,0.000000
116946,Roger Federer,1.0,0.000000
116965,Roger Federer,1.0,0.000000
116993,Roger Federer,1.0,0.000000
116998,Roger Federer,1.0,0.000000
117011,Roger Federer,1.0,0.000000
120431,Roger Federer,13.0,0.615385
120462,Roger Federer,13.0,0.615385
139371,Roger Federer,9.0,0.333333


## *Serve Statistics*


In [97]:
from collections import defaultdict, deque
import numpy as np
import pandas as pd

# Shared cutoff for BOTH Serve Statistics and Return Statistics.
SERVE_RETURN_HISTORY_LAG_DAYS = 10
assert SERVE_RETURN_HISTORY_LAG_DAYS == MATCHUP_HISTORY_LAG_DAYS
assert SERVE_RETURN_HISTORY_LAG_DAYS == ELO_HISTORY_LAG_DAYS
assert SERVE_RETURN_HISTORY_LAG_DAYS == RECENT_HISTORY_LAG_DAYS

CAREER_SERVE_COLUMNS = [
    f"player{side}_{suffix}"
    for side in ("A", "B")
    for suffix in (
        "ace_pct", "df_pct", "1stIn_pct", "1stWon_pct", "2ndWon_pct",
        "career_p", "serve_matches",
    )
]


def _history_events(frame, requested_columns, lag_days=SERVE_RETURN_HISTORY_LAG_DAYS):
    """Yield ('release', historical row) and ('snapshot', current row) events.

    Release happens ONLY when the historical tournament date is <= T - lag.
    This is streaming, and does not assume any ordering among same-date rows.
    """
    if not pd.api.types.is_datetime64_any_dtype(frame["tourney_date"]):
        raise TypeError("tourney_date must be a pandas datetime column")
    if frame["tourney_date"].isna().any() or not frame["tourney_date"].is_monotonic_increasing:
        raise ValueError("Historical input must have nonmissing, sorted tourney_date")
    if frame["match_id"].duplicated().any():
        raise ValueError("Each match_id must be unique")

    columns = list(dict.fromkeys(["match_id", "tourney_date", *requested_columns]))
    lag = pd.Timedelta(days=lag_days)
    pending = deque()

    for current in frame[columns].itertuples(index=False, name="HistoricalMatch"):
        cutoff = current.tourney_date - lag
        while pending and pending[0].tourney_date <= cutoff:
            yield "release", pending.popleft()
        yield "snapshot", current
        pending.append(current)


def _valid_server(row, side):
    """Reject missing/corrupt service side before it enters any cumulative total."""
    names = ("ace", "df", "svpt", "1stIn", "1stWon", "2ndWon")
    values = [getattr(row, f"player{side}_{name}") for name in names]
    try:
        if not all(pd.notna(v) and np.isfinite(v) for v in values):
            return False
    except TypeError:
        return False
    ace, df, svpt, first_in, first_won, second_won = values
    return (
        svpt > 0 and 0 <= first_in <= svpt
        and 0 <= first_won <= first_in
        and 0 <= second_won <= svpt - first_in
        and 0 <= ace <= svpt and 0 <= df <= svpt - first_in
    )


def _valid_break_points(row, side):
    faced = getattr(row, f"player{side}_bpFaced")
    saved = getattr(row, f"player{side}_bpSaved")
    try:
        return (
            pd.notna(faced) and pd.notna(saved)
            and np.isfinite(faced) and np.isfinite(saved)
            and 0 <= saved <= faced
        )
    except TypeError:
        return False


def _attach_historical_features(model_frame, feature_frame, feature_columns):
    lookup = feature_frame.set_index("match_id")
    if not model_frame["match_id"].isin(lookup.index).all():
        raise ValueError("Historical feature rows missing from modeling frame")
    model_frame[feature_columns] = lookup.loc[
        model_frame["match_id"], feature_columns
    ].to_numpy()


def add_career_serve_features(frame, lag_days=SERVE_RETURN_HISTORY_LAG_DAYS):
    stats = defaultdict(lambda: dict(ace=0, df=0, svpt=0,
                                     first_in=0, first_won=0,
                                     second_won=0, matches=0))
    out = {col: [] for col in CAREER_SERVE_COLUMNS}
    required = [f"player{side}_{name}" for side in ("A", "B")
                for name in ("id", "ace", "df", "svpt", "1stIn", "1stWon", "2ndWon")]

    def values(s):
        if not s["svpt"]:
            return (np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, s["matches"])
        return (
            s["ace"] / s["svpt"],
            s["df"] / s["svpt"],
            s["first_in"] / s["svpt"],
            s["first_won"] / s["first_in"] if s["first_in"] else np.nan,
            s["second_won"] / (s["svpt"] - s["first_in"])
                if s["svpt"] > s["first_in"] else np.nan,
            (s["first_won"] + s["second_won"]) / s["svpt"],
            s["matches"],
        )

    for event, row in _history_events(frame, required, lag_days):
        if event == "release":
            for side in ("A", "B"):
                if not _valid_server(row, side):
                    continue
                s = stats[getattr(row, f"player{side}_id")]
                for field, suffix in (("ace", "ace"), ("df", "df"),
                                      ("svpt", "svpt"), ("first_in", "1stIn"),
                                      ("first_won", "1stWon"), ("second_won", "2ndWon")):
                    s[field] += getattr(row, f"player{side}_{suffix}")
                s["matches"] += 1
        else:
            for side in ("A", "B"):
                s = stats[getattr(row, f"player{side}_id")]
                for name, value in zip(("ace_pct", "df_pct", "1stIn_pct", "1stWon_pct",
                                        "2ndWon_pct", "career_p", "serve_matches"), values(s)):
                    out[f"player{side}_{name}"].append(value)

    result = pd.DataFrame(out)
    result.insert(0, "match_id", frame["match_id"].to_numpy())
    return result


career_serve_full = add_career_serve_features(matches_df_fixed_copy)
_attach_historical_features(matches_df, career_serve_full, CAREER_SERVE_COLUMNS)
print("Attached", len(CAREER_SERVE_COLUMNS), "career serve columns")


Attached 14 career serve columns


In [98]:
#test-1
matches_df[["playerA_name", "playerA_serve_matches", "playerA_career_p"]].tail(15)

,playerA_name,playerA_serve_matches,playerA_career_p
855115,Andres Martin,141.0,0.611083
855116,Timo Legout,24.0,0.567313
855117,Liam Draxl,193.0,0.615671
855118,Braden Shick,67.0,0.640455
855119,Hayato Matsuoka,91.0,0.611531
855120,Karl Poling,97.0,0.599042
855121,Edas Butvilas,152.0,0.632042
855122,Yuta Shimizu,237.0,0.606240
855123,Timo Legout,24.0,0.567313
855124,Liam Draxl,193.0,0.615671


In [99]:
# Head-to-head service-point history. A server/opponent directional key
# replaces the old sorted-pair bookkeeping without changing the features.
MATCHUP_SERVE_COLUMNS = [
    f"player{side}_{name}" for side in ("A", "B")
    for name in ("matchup_p", "matchup_service_points")
]


def add_matchup_serve_features(frame, lag_days=SERVE_RETURN_HISTORY_LAG_DAYS):
    matchup = defaultdict(lambda: [0, 0])  # (server, opponent) -> [won, served]
    out = {name: [] for name in MATCHUP_SERVE_COLUMNS}
    required = [f"player{side}_{name}" for side in ("A", "B")
                for name in ("id", "ace", "df", "svpt", "1stIn", "1stWon", "2ndWon")]

    for event, row in _history_events(frame, required, lag_days):
        playerA, playerB = row.playerA_id, row.playerB_id
        if event == "release":
            for side, server, opponent in (("A", playerA, playerB), ("B", playerB, playerA)):
                if _valid_server(row, side):
                    record = matchup[(server, opponent)]
                    record[0] += (getattr(row, f"player{side}_1stWon")
                                  + getattr(row, f"player{side}_2ndWon"))
                    record[1] += getattr(row, f"player{side}_svpt")
        else:
            for side, server, opponent in (("A", playerA, playerB), ("B", playerB, playerA)):
                won, served = matchup[(server, opponent)]
                out[f"player{side}_matchup_p"].append(won / served if served else np.nan)
                out[f"player{side}_matchup_service_points"].append(served)

    result = pd.DataFrame(out)
    result.insert(0, "match_id", frame["match_id"].to_numpy())
    return result


matchup_serve_full = add_matchup_serve_features(matches_df_fixed_copy)
_attach_historical_features(matches_df, matchup_serve_full, MATCHUP_SERVE_COLUMNS)
print("Attached", len(MATCHUP_SERVE_COLUMNS), "matchup serve columns")


Attached 4 matchup serve columns


In [100]:
#test-2
test = matches_df[
    (
        (matches_df["playerA_name"] == "Roger Federer")
        & (matches_df["playerB_name"] == "Rafael Nadal")
    )
    |
    (
        (matches_df["playerA_name"] == "Rafael Nadal")
        & (matches_df["playerB_name"] == "Roger Federer")
    )
]

test[
    [
        "playerA_name",
        "playerB_name",
        "playerA_matchup_p",
        "playerB_matchup_p",
        "playerA_matchup_service_points",
        "playerB_matchup_service_points"
    ]
].tail(20)

,playerA_name,playerB_name,playerA_matchup_p,playerB_matchup_p,playerA_matchup_service_points,playerB_matchup_service_points
417410,Rafael Nadal,Roger Federer,0.630340,0.620978,2294.0,2269.0
423186,Roger Federer,Rafael Nadal,0.625804,0.630932,2333.0,2360.0
425562,Rafael Nadal,Roger Federer,0.633680,0.623588,2405.0,2391.0
427572,Rafael Nadal,Roger Federer,0.634078,0.620690,2506.0,2494.0
443473,Rafael Nadal,Roger Federer,0.632868,0.618702,2653.0,2620.0
445010,Rafael Nadal,Roger Federer,0.630290,0.621429,2694.0,2660.0
448420,Rafael Nadal,Roger Federer,0.632451,0.621467,2835.0,2795.0
475829,Rafael Nadal,Roger Federer,0.630578,0.621887,2891.0,2851.0
480785,Rafael Nadal,Roger Federer,0.631668,0.618934,2943.0,2926.0
490106,Roger Federer,Rafael Nadal,0.615979,0.632831,2979.0,2985.0


In [101]:
# Surface-specific serve point probability, using the same delayed history.
SURFACE_SERVE_COLUMNS = [
    f"player{side}_{name}" for side in ("A", "B")
    for name in ("surface_p", "surface_serve_matches")
]


def _normalize_surface(surface):
    if pd.isna(surface) or str(surface).strip() == "":
        return None
    return str(surface).strip().capitalize()


def add_surface_serve_features(frame, lag_days=SERVE_RETURN_HISTORY_LAG_DAYS):
    stats = defaultdict(lambda: [0, 0, 0])  # won, served, matches
    out = {name: [] for name in SURFACE_SERVE_COLUMNS}
    required = ["surface"] + [f"player{side}_{name}" for side in ("A", "B")
                for name in ("id", "ace", "df", "svpt", "1stIn", "1stWon", "2ndWon")]

    for event, row in _history_events(frame, required, lag_days):
        surface = _normalize_surface(row.surface)
        if event == "release":
            if surface is None:
                continue  # no unknown-surface observations in surface history
            for side in ("A", "B"):
                if not _valid_server(row, side):
                    continue
                s = stats[(getattr(row, f"player{side}_id"), surface)]
                s[0] += getattr(row, f"player{side}_1stWon") + getattr(row, f"player{side}_2ndWon")
                s[1] += getattr(row, f"player{side}_svpt")
                s[2] += 1
        else:
            for side in ("A", "B"):
                s = stats[(getattr(row, f"player{side}_id"), surface)] if surface is not None else (0, 0, 0)
                out[f"player{side}_surface_p"].append(s[0] / s[1] if s[1] else np.nan)
                out[f"player{side}_surface_serve_matches"].append(s[2])

    result = pd.DataFrame(out)
    result.insert(0, "match_id", frame["match_id"].to_numpy())
    return result


surface_serve_full = add_surface_serve_features(matches_df_fixed_copy)
_attach_historical_features(matches_df, surface_serve_full, SURFACE_SERVE_COLUMNS)
print("Attached", len(SURFACE_SERVE_COLUMNS), "surface serve columns")


Attached 4 surface serve columns


In [102]:
#test-3
matches_df[["playerA_career_p", "playerA_surface_p"]].sample(15)

,playerA_career_p,playerA_surface_p
372307,NaN,NaN
313031,NaN,NaN
249206,NaN,NaN
252327,0.474576,0.474576
27829,NaN,NaN
757568,0.625226,0.625328
22815,NaN,NaN
657268,0.536680,0.565657
137788,0.593407,NaN
546360,0.597048,0.605778


## *Return Statistics*


In [103]:
RETURN_FEATURE_COLUMNS = [
    f"player{side}_{name}" for side in ("A", "B")
    for name in ("return_pct", "bp_convert_pct", "return_matches")
]


def add_return_features(frame, lag_days=SERVE_RETURN_HISTORY_LAG_DAYS):
    stats = defaultdict(lambda: dict(return_won=0, return_played=0,
                                     bp_converted=0, bp_opportunities=0,
                                     matches=0))
    out = {name: [] for name in RETURN_FEATURE_COLUMNS}
    required = [f"player{side}_{name}" for side in ("A", "B")
                for name in ("id", "ace", "df", "svpt", "1stIn", "1stWon",
                             "2ndWon", "bpFaced", "bpSaved")]

    for event, row in _history_events(frame, required, lag_days):
        if event == "release":
            # A returns against B's serve; B returns against A's serve.
            for return_side, server_side in (("A", "B"), ("B", "A")):
                s = stats[getattr(row, f"player{return_side}_id")]
                if _valid_server(row, server_side):
                    played = getattr(row, f"player{server_side}_svpt")
                    service_won = (getattr(row, f"player{server_side}_1stWon")
                                   + getattr(row, f"player{server_side}_2ndWon"))
                    s["return_won"] += played - service_won
                    s["return_played"] += played
                    s["matches"] += 1
                if _valid_break_points(row, server_side):
                    faced = getattr(row, f"player{server_side}_bpFaced")
                    saved = getattr(row, f"player{server_side}_bpSaved")
                    s["bp_converted"] += faced - saved
                    s["bp_opportunities"] += faced
        else:
            for side in ("A", "B"):
                s = stats[getattr(row, f"player{side}_id")]
                out[f"player{side}_return_pct"].append(
                    s["return_won"] / s["return_played"] if s["return_played"] else 0.5
                )
                out[f"player{side}_bp_convert_pct"].append(
                    s["bp_converted"] / s["bp_opportunities"] if s["bp_opportunities"] else 0.5
                )
                out[f"player{side}_return_matches"].append(s["matches"])

    result = pd.DataFrame(out)
    result.insert(0, "match_id", frame["match_id"].to_numpy())
    return result


return_full = add_return_features(matches_df_fixed_copy)
_attach_historical_features(matches_df, return_full, RETURN_FEATURE_COLUMNS)
print("Attached", len(RETURN_FEATURE_COLUMNS), "return-statistics columns")


Attached 6 return-statistics columns


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\4182400259.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_frame[feature_columns] = lookup.loc[


In [104]:
#test-1
matches_df[["playerA_return_pct", "playerB_return_pct"]].describe()

,playerA_return_pct,playerB_return_pct
count,855077.000000,855077.000000
mean,0.415669,0.415718
std,0.073192,0.073178
min,0.000000,0.030303
25%,0.366306,0.366303
50%,0.398380,0.398419
75%,0.500000,0.500000
max,0.714286,0.714286


In [105]:
# V1.1 Serve & Return synthetic regression tests (no source CSV required).
from pandas.testing import assert_frame_equal


def _sr_match(mid, date, surface, A, B, A_stats, B_stats):
    row = dict(match_id=mid, tourney_date=pd.Timestamp(date), surface=surface,
               playerA_id=A, playerB_id=B)
    for side, values in (("A", A_stats), ("B", B_stats)):
        for name, value in zip(
            ("ace", "df", "svpt", "1stIn", "1stWon", "2ndWon", "bpFaced", "bpSaved"),
            values,
        ):
            row[f"player{side}_{name}"] = value
    return row


# Each tuple is ace, df, service points, first serves in, first won,
# second won, break points faced, break points saved.
_good = (2, 1, 20, 12, 8, 4, 2, 1)
_other = (1, 1, 20, 10, 6, 6, 3, 2)
_bad_bp = (1, 1, 20, 10, 6, 5, 3, -2)
_missing = (np.nan,) * 8
_sr_frame = pd.DataFrame([
    _sr_match(0, "2020-01-01", "Hard", 1, 2, _good, _other),
    _sr_match(1, "2020-01-01", np.nan, 2, 1, _bad_bp, _good),
    _sr_match(2, "2020-01-09", "Hard", 1, 3, _good, _other),
    _sr_match(3, "2020-01-11", "Hard", 1, 2, _good, _other),
    # One missing service side; the other side should still contribute.
    _sr_match(4, "2020-01-12", "Clay", 1, 2, _good, _missing),
    _sr_match(5, "2020-01-25", "Hard", 1, 2, _good, _other),
])

_sr_funcs = [add_career_serve_features, add_matchup_serve_features,
             add_surface_serve_features, add_return_features]
_sr_cols_all = [*CAREER_SERVE_COLUMNS, *MATCHUP_SERVE_COLUMNS,
                *SURFACE_SERVE_COLUMNS, *RETURN_FEATURE_COLUMNS]


def _sr_predict(frame):
    pieces = [func(frame).set_index("match_id") for func in _sr_funcs]
    return pd.concat(pieces, axis=1).sort_index()


_sr_out = _sr_predict(_sr_frame)
assert _sr_out.loc[2, "playerA_serve_matches"] == 0  # Jan 1 only 8 days earlier
assert _sr_out.loc[3, "playerA_serve_matches"] == 2  # Jan 1 now exactly 10 days old
assert _sr_out.loc[3, "playerA_surface_serve_matches"] == 1  # one unknown surface
assert _sr_out.loc[3, "playerA_matchup_service_points"] == 40  # both directions
assert _sr_out.loc[3, "playerA_return_matches"] == 2
assert np.isclose(_sr_out.loc[3, "playerA_career_p"], 24 / 40)
assert 0 <= _sr_out.loc[3, "playerA_bp_convert_pct"] <= 1  # invalid bpSaved skipped
assert _sr_out.loc[5, "playerA_serve_matches"] == 5  # valid service sides count
assert _sr_out.loc[5, "playerB_serve_matches"] == 3  # missing B side not counted

_sr_shuffled = _sr_frame.sample(frac=1, random_state=43).sort_values(
    "tourney_date", kind="mergesort"
)
assert_frame_equal(_sr_out[_sr_cols_all], _sr_predict(_sr_shuffled)[_sr_cols_all],
                   check_dtype=False)
print("Serve/return delayed cutoff, missing-data and order-invariance tests passed.")


Serve/return delayed cutoff, missing-data and order-invariance tests passed.


## *Fatigue Features*


In [106]:
# V1.1 Fatigue / workload: delayed historical eligibility and date-based windows.
from collections import defaultdict, deque
import numpy as np
import pandas as pd

FATIGUE_HISTORY_LAG_DAYS = 10
FATIGUE_WINDOW_DAYS = {"matches_30d": 30, "matches_60d": 60,
                       "tournaments_5w": 35, "tournaments_10w": 70}
assert FATIGUE_HISTORY_LAG_DAYS == MATCHUP_HISTORY_LAG_DAYS
assert FATIGUE_HISTORY_LAG_DAYS == RECENT_HISTORY_LAG_DAYS
assert FATIGUE_HISTORY_LAG_DAYS == ELO_HISTORY_LAG_DAYS
assert FATIGUE_HISTORY_LAG_DAYS == SERVE_RETURN_HISTORY_LAG_DAYS

FATIGUE_SUFFIXES = (
    "matches_30d", "wins_30d", "losses_30d", "winpct_30d",
    "matches_60d", "tournaments_5w", "tournaments_10w",
)
FATIGUE_FEATURE_COLUMNS = [
    f"player{side}_{suffix}" for side in ("A", "B") for suffix in FATIGUE_SUFFIXES
]
OLD_FATIGUE_7D_COLUMNS = [
    f"player{side}_{suffix}_7d"
    for side in ("A", "B")
    for suffix in ("matches", "wins", "losses", "winpct")
]


def _fatigue_event_key(row):
    """Identify a tournament edition, not each individual match/round."""
    if pd.notna(row.tourney_id) and str(row.tourney_id).strip():
        return ("id", str(row.tourney_id), row.tourney_date)
    if pd.notna(row.tourney_name) and str(row.tourney_name).strip():
        level = row.tourney_level if pd.notna(row.tourney_level) else None
        return ("fallback", str(row.tourney_name).strip(), level, row.tourney_date)
    return None


def add_fatigue_features(frame, history_lag_days=FATIGUE_HISTORY_LAG_DAYS):
    """Compute date-frozen workload features from the complete match history.

    Missing tournament IDs cannot inflate counts: a date/name fallback is used.
    Match rows without valid player IDs or a binary outcome are not added
    to history. The calculation never relies on within-date match ordering.
    """
    required = (
        "match_id", "tourney_date", "tourney_id", "tourney_name",
        "tourney_level", "playerA_id", "playerB_id", "playerA_wins",
    )
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f"Missing fatigue input columns: {missing}")

    source = frame.loc[:, required].copy()
    if not pd.api.types.is_datetime64_any_dtype(source["tourney_date"]):
        source["tourney_date"] = pd.to_datetime(source["tourney_date"], errors="raise")
    if source["tourney_date"].isna().any():
        raise ValueError("Fatigue features require nonmissing tournament dates")
    if not source["tourney_date"].is_monotonic_increasing:
        raise ValueError("Sort by tourney_date before generating fatigue features")
    if source["match_id"].isna().any() or not source["match_id"].is_unique:
        raise ValueError("Fatigue features require unique, non-null match_id values")

    # Eligible historical results are released from this queue only after lag.
    pending = deque()                    # (date, player A, player B, A won, event ID)
    # Each player's history contains only recently released observations.
    history = defaultdict(deque)         # (reference date, won (0/1), event ID)
    features = {name: [] for name in FATIGUE_FEATURE_COLUMNS}
    previous_date = None
    cutoff = None
    oldest_70d = None
    oldest_60d = None
    oldest_35d = None
    oldest_30d = None

    def release(record):
        date, a, b, a_wins, event = record
        if (pd.isna(a) or pd.isna(b) or a == b
                or pd.isna(a_wins) or a_wins not in (0, 1)):
            return
        history[a].append((date, int(a_wins), event))
        history[b].append((date, 1 - int(a_wins), event))

    for row in source.itertuples(index=False):
        date = row.tourney_date
        if previous_date is None or date != previous_date:
            cutoff = date - pd.Timedelta(days=history_lag_days)
            oldest_70d = cutoff - pd.Timedelta(days=70)
            oldest_60d = cutoff - pd.Timedelta(days=60)
            oldest_35d = cutoff - pd.Timedelta(days=35)
            oldest_30d = cutoff - pd.Timedelta(days=30)
            previous_date = date

        # Release old matches BEFORE generating any features for this date.
        while pending and pending[0][0] <= cutoff:
            release(pending.popleft())

        for side in ("A", "B"):
            player = getattr(row, f"player{side}_id")
            records = history[player] if pd.notna(player) else ()
            if pd.notna(player):
                while records and records[0][0] <= oldest_70d:
                    records.popleft()

            matches_30d = wins_30d = matches_60d = 0
            played_5w, played_10w = set(), set()
            for match_date, won, tournament in records:
                # The queue holds only eligible histories; the boundaries
                # below are strict at their start and inclusive at cutoff.
                if match_date > oldest_60d:
                    matches_60d += 1
                if match_date > oldest_30d:
                    matches_30d += 1
                    wins_30d += won
                if tournament is not None:
                    if match_date > oldest_35d:
                        played_5w.add(tournament)
                    played_10w.add(tournament)

            values = (
                matches_30d, wins_30d, matches_30d - wins_30d,
                wins_30d / matches_30d if matches_30d else 0.5,
                matches_60d, len(played_5w), len(played_10w),
            )
            for suffix, value in zip(FATIGUE_SUFFIXES, values):
                features[f"player{side}_{suffix}"].append(value)

        pending.append((
            date, row.playerA_id, row.playerB_id,
            row.playerA_wins, _fatigue_event_key(row),
        ))

    output = pd.DataFrame(features)
    output.insert(0, "match_id", source["match_id"].to_numpy())
    return output


# Use complete history, including unknown-surface matches, but attach only
# to the surviving modeling rows. Do not reintroduce unknown-surface matches.
fatigue_full = add_fatigue_features(matches_df_fixed_copy)
# Also remove 7-day columns if this cell is rerun in an existing kernel.
matches_df.drop(columns=OLD_FATIGUE_7D_COLUMNS, errors="ignore", inplace=True)
_attach_historical_features(matches_df, fatigue_full, FATIGUE_FEATURE_COLUMNS)
print("Attached", len(FATIGUE_FEATURE_COLUMNS), "fatigue/workload columns")


Attached 14 fatigue/workload columns


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\4182400259.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_frame[feature_columns] = lookup.loc[


In [107]:
# V1.1 fatigue synthetic tests — no ATP source CSV required.
# Use date reference boundaries rather than presumed actual match dates.
def _check_fatigue_features():
    reference = pd.Timestamp("2026-05-01")
    eligible = reference - pd.Timedelta(days=FATIGUE_HISTORY_LAG_DAYS)

    def example(mid, days_from_cutoff, a, b, a_wins, event):
        date = eligible + pd.Timedelta(days=days_from_cutoff)
        return dict(match_id=mid, tourney_date=date, tourney_id=event,
                    tourney_name=event, tourney_level="A", playerA_id=a,
                    playerB_id=b, playerA_wins=a_wins, surface="Hard")

    sample = pd.DataFrame([
        example(1, -70, 1, 101, 1, "outside10w"),  # strict lower boundary
        example(2, -69, 1, 102, 0, "inside10w"),
        example(3, -60, 1, 103, 1, "outside60d"),
        example(4, -59, 1, 104, 0, "inside60d"),
        example(5, -35, 1, 105, 1, "outside5w"),
        example(6, -34, 1, 106, 0, "inside5w"),
        example(7, -30, 1, 107, 1, "outside30d"),
        example(8, -29, 1, 108, 0, "inside30d"),
        example(9,   0, 1, 109, 1, "atcutoff"),
        example(10,  1, 1, 110, 0, "notyet"),
        example(11, 10, 1, 111, 0, "current"),
        example(12, 10, 1, 112, 1, "current"),
    ]).sort_values("tourney_date", kind="stable").reset_index(drop=True)

    result = add_fatigue_features(sample).set_index("match_id")
    current = result.loc[11]
    assert current["playerA_matches_30d"] == 2
    assert current["playerA_wins_30d"] == 1
    assert current["playerA_losses_30d"] == 1
    assert current["playerA_winpct_30d"] == 0.5
    assert current["playerA_matches_60d"] == 6
    assert current["playerA_tournaments_5w"] == 4
    assert current["playerA_tournaments_10w"] == 8
    assert result.loc[11, FATIGUE_FEATURE_COLUMNS].equals(
        result.loc[12, FATIGUE_FEATURE_COLUMNS]
    )  # same-day matches cannot see each other

    # Shuffle only the equal-date group: all outputs must remain unchanged.
    reversed_same_day = pd.concat([sample.iloc[:-2], sample.iloc[-2:][::-1]])
    other = add_fatigue_features(reversed_same_day).set_index("match_id")
    pd.testing.assert_frame_equal(result.sort_index(), other.sort_index())

    # A match without a surface still contributes to overall workload.
    missing_surface = sample.copy()
    missing_surface.loc[missing_surface.match_id == 9, "surface"] = None
    pd.testing.assert_frame_equal(
        add_fatigue_features(sample), add_fatigue_features(missing_surface)
    )

    # A missing outcome cannot enter historical wins/match counts.
    missing_result = sample.copy()
    missing_result.loc[missing_result.match_id == 9, "playerA_wins"] = np.nan
    updated = add_fatigue_features(missing_result).set_index("match_id")
    assert updated.loc[11, "playerA_matches_30d"] == 1

    print("Fatigue synthetic tests passed: cutoff, 30/60-day and 5/10-week boundaries,",
          "same-date invariance, missing surface and missing outcome")


_check_fatigue_features()


Fatigue synthetic tests passed: cutoff, 30/60-day and 5/10-week boundaries, same-date invariance, missing surface and missing outcome


In [108]:
# V1.1 fatigue sanity checks: no seven-day fields, count consistency,
# nonnegative activity, and 30-day workload contained in 60-day workload.
assert not set(OLD_FATIGUE_7D_COLUMNS).intersection(matches_df.columns)
assert matches_df["playerA_matches_30d"].ge(0).all()
assert matches_df["playerB_matches_30d"].ge(0).all()
for side in ("A", "B"):
    assert matches_df[f"player{side}_matches_30d"].le(
        matches_df[f"player{side}_matches_60d"]
    ).all()
    assert (
        matches_df[f"player{side}_wins_30d"]
        + matches_df[f"player{side}_losses_30d"]
        == matches_df[f"player{side}_matches_30d"]
    ).all()
    assert matches_df[f"player{side}_tournaments_5w"].le(
        matches_df[f"player{side}_tournaments_10w"]
    ).all()
    assert matches_df[f"player{side}_winpct_30d"].between(0, 1).all()

matches_df[["playerA_matches_30d", "playerA_wins_30d",
            "playerA_winpct_30d", "playerA_matches_60d",
            "playerA_tournaments_5w", "playerA_tournaments_10w"]].tail(20)


,playerA_matches_30d,playerA_wins_30d,playerA_winpct_30d,playerA_matches_60d,playerA_tournaments_5w,playerA_tournaments_10w
855110,2.0,0.0,0.000000,9.0,2.0,6.0
855111,11.0,8.0,0.727273,16.0,3.0,8.0
855112,7.0,4.0,0.571429,17.0,3.0,8.0
855113,0.0,0.0,0.500000,2.0,1.0,3.0
855114,3.0,1.0,0.333333,11.0,3.0,6.0
855115,3.0,1.0,0.333333,8.0,3.0,8.0
855116,0.0,0.0,0.500000,3.0,1.0,1.0
855117,1.0,0.0,0.000000,11.0,2.0,6.0
855118,7.0,6.0,0.857143,14.0,3.0,6.0
855119,6.0,4.0,0.666667,12.0,3.0,6.0


## *Rank Features*

In [109]:
# These flags use only the recorded pre-match rank/points, with no fitted preprocessing.
# Missing rank does not necessarily mean the player was truly unranked.
matches_df["playerA_is_unranked"] = matches_df["playerA_rank"].isna().astype("int8")
matches_df["playerB_is_unranked"] = matches_df["playerB_rank"].isna().astype("int8")

matches_df["playerA_rank_points_missing"] = matches_df["playerA_rank_points"].isna().astype("int8")
matches_df["playerB_rank_points_missing"] = matches_df["playerB_rank_points"].isna().astype("int8")


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\84225933.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["playerA_is_unranked"] = matches_df["playerA_rank"].isna().astype("int8")
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\84225933.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["playerB_is_unranked"] = matches_df["playerB_rank"].isna().astype("int8")
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\84225933.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

In [110]:
# V1.1: DEFER missing-rank imputation until after the temporal train/val/test split.
# In v1_DataAnal_DataPrep.ipynb: learn MAX_RANK on TRAIN ONLY, define
# UNRANKED_RANK = MAX_RANK + 1, and fill all three splits with that constant.
# Retain these two column names for downstream-model compatibility; NaN is
# intentional here and will be filled by the downstream fitted transformer.
matches_df["playerA_rank_filled"] = matches_df["playerA_rank"].copy()
matches_df["playerB_rank_filled"] = matches_df["playerB_rank"].copy()


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\294723377.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["playerA_rank_filled"] = matches_df["playerA_rank"].copy()
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\294723377.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["playerB_rank_filled"] = matches_df["playerB_rank"].copy()


In [111]:
# Only observed ranks contribute to provisional differences here.
# IMPORTANT: Recompute both features AFTER training-only imputation downstream;
# otherwise matches with missing ranks would keep NaN values unnecessarily.
matches_df["rank_diff"] = matches_df["playerA_rank_filled"] - matches_df["playerB_rank_filled"]
matches_df["rank_ratio"] = matches_df["playerA_rank_filled"] / matches_df["playerB_rank_filled"]


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1182766419.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["rank_diff"] = matches_df["playerA_rank_filled"] - matches_df["playerB_rank_filled"]
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1182766419.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["rank_ratio"] = matches_df["playerA_rank_filled"] / matches_df["playerB_rank_filled"]


In [112]:
# V1.1: DEFER ranking-points median imputation until AFTER the temporal split.
# Learn the pooled player A/B points median using TRAIN ONLY, then use that
# same median for train, validation and test. Recompute rank_points_diff from
# the imputed values in v1_DataAnal_DataPrep.ipynb.
# Until then, leave NaN wherever either raw points value is missing.
matches_df["rank_points_diff"] = (
    matches_df["playerA_rank_points"] - matches_df["playerB_rank_points"]
)


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\3645907291.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df["rank_points_diff"] = (


In [113]:
# Threshold indicators do not require rank imputation.
# Missing ranks simply do not satisfy the top-K thresholds (and are flagged
# separately above). Using raw ranks avoids dependence on future data.
for threshold in [10, 32, 64, 100]:
    matches_df[f"playerA_top{threshold}"] = matches_df["playerA_rank"].le(threshold).astype("int8")
    matches_df[f"playerB_top{threshold}"] = matches_df["playerB_rank"].le(threshold).astype("int8")


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\548670014.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df[f"playerA_top{threshold}"] = matches_df["playerA_rank"].le(threshold).astype("int8")
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\548670014.py:6: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df[f"playerB_top{threshold}"] = matches_df["playerB_rank"].le(threshold).astype("int8")
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\548670014.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usu

In [114]:
#test-1

matches_df[[
    "playerA_is_unranked",
    "playerB_rank_points_missing",
    "playerA_rank_filled", 
    "playerA_rank", 
    "rank_diff",
    "rank_ratio",
    "rank_points_diff",
    "playerA_top32"
]
    
].sample(15)

,playerA_is_unranked,playerB_rank_points_missing,playerA_rank_filled,playerA_rank,rank_diff,rank_ratio,rank_points_diff,playerA_top32
415714,0,0,459.0,459.0,-186.0,0.711628,46.0,0
114093,1,1,NaN,NaN,NaN,NaN,NaN,0
52913,0,0,763.0,763.0,361.0,1.898010,-48.0,0
836512,0,0,611.0,611.0,189.0,1.447867,-48.0,0
126207,0,0,550.0,550.0,157.0,1.399491,-37.0,0
831742,1,0,NaN,NaN,NaN,NaN,NaN,0
597182,0,0,159.0,159.0,123.0,4.416667,-869.0,0
579497,0,0,608.0,608.0,-374.0,0.619145,34.0,0
587496,0,0,798.0,798.0,-8.0,0.990074,1.0,0
596893,0,0,248.0,248.0,-619.0,0.286044,192.0,0


## *Physical features*

In [115]:
# V1.1: Defer AGE median imputation until AFTER the temporal train/validation/test split.
# Learn the pooled player A/B age median using TRAIN ONLY in v1_DataAnal_DataPrep.ipynb.
# Apply that fixed value to all three splits, then RECOMPUTE age_diff there.
# Preserve the original ages and NaNs in this upstream feature-generation notebook.
matches_df['age_diff'] = matches_df['playerA_age'] - matches_df['playerB_age']

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1700286732.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['age_diff'] = matches_df['playerA_age'] - matches_df['playerB_age']


In [116]:
# V1.1: Defer HEIGHT median imputation until AFTER the temporal train/validation/test split.
# Learn the pooled player A/B height median using TRAIN ONLY in v1_DataAnal_DataPrep.ipynb.
# Apply that fixed value to all three splits, then RECOMPUTE height_diff there.
# Preserve the original heights and NaNs in this upstream feature-generation notebook.
matches_df['height_diff'] = matches_df['playerA_ht'] - matches_df['playerB_ht']

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1586650530.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['height_diff'] = matches_df['playerA_ht'] - matches_df['playerB_ht']


## *Seed features*

In [117]:
matches_df['playerA_seeded'] = matches_df['playerA_seed'].notna().astype('int8')
matches_df['playerB_seeded'] = matches_df['playerB_seed'].notna().astype('int8')

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\672625524.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerA_seeded'] = matches_df['playerA_seed'].notna().astype('int8')
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\672625524.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerB_seeded'] = matches_df['playerB_seed'].notna().astype('int8')


In [118]:
matches_df['seed_diff'] = np.where(
    matches_df['playerA_seed'].notna() & matches_df['playerB_seed'].notna(), 
    matches_df['playerA_seed'] - matches_df['playerB_seed'], 
    np.nan
)

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\273368266.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['seed_diff'] = np.where(


In [119]:
matches_df['seed_diff_missing'] = matches_df['seed_diff'].isna().astype('int8')

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\995359959.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['seed_diff_missing'] = matches_df['seed_diff'].isna().astype('int8')


## *Surface Specialization features*

In [120]:
#win rate
matches_df['playerA_win_rate_specialization'] = matches_df["playerA_surface_career_pct"] - matches_df['playerA_career_pct']
matches_df['playerB_win_rate_specialization'] = matches_df['playerB_surface_career_pct'] - matches_df['playerB_career_pct']

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1546508238.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerA_win_rate_specialization'] = matches_df["playerA_surface_career_pct"] - matches_df['playerA_career_pct']
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\1546508238.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerB_win_rate_specialization'] = matches_df['playerB_surface_career_pct'] - matches_df['playerB_career_pct']


In [121]:
#elo
matches_df['playerA_elo_specialization'] = matches_df['playerA_surface_elo'] - matches_df['playerA_elo']
matches_df['playerB_elo_specialization'] = matches_df['playerB_surface_elo'] - matches_df['playerB_elo']

C:\Users\HP\AppData\Local\Temp\ipykernel_3604\167736039.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerA_elo_specialization'] = matches_df['playerA_surface_elo'] - matches_df['playerA_elo']
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\167736039.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerB_elo_specialization'] = matches_df['playerB_surface_elo'] - matches_df['playerB_elo']


In [122]:
#p
matches_df['playerA_serve_specialization'] = matches_df['playerA_surface_p'] - matches_df['playerA_career_p']
matches_df['playerB_serve_specialization'] = matches_df['playerB_surface_p'] - matches_df['playerB_career_p']


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\221088805.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerA_serve_specialization'] = matches_df['playerA_surface_p'] - matches_df['playerA_career_p']
C:\Users\HP\AppData\Local\Temp\ipykernel_3604\221088805.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matches_df['playerB_serve_specialization'] = matches_df['playerB_surface_p'] - matches_df['playerB_career_p']


## *Experience*


In [123]:
# V1.1 Experience: historical, order-invariant counters at a 10-day cutoff.
# Retain all ten existing non-duration features; add two distinct-edition counts.
# The six Surface Specialization formulas in cells above are unchanged.
from collections import defaultdict, deque
import numpy as np
import pandas as pd

EXPERIENCE_HISTORY_LAG_DAYS = 10
assert EXPERIENCE_HISTORY_LAG_DAYS == MATCHUP_HISTORY_LAG_DAYS
assert EXPERIENCE_HISTORY_LAG_DAYS == ELO_HISTORY_LAG_DAYS
assert EXPERIENCE_HISTORY_LAG_DAYS == RECENT_HISTORY_LAG_DAYS
assert EXPERIENCE_HISTORY_LAG_DAYS == SERVE_RETURN_HISTORY_LAG_DAYS
assert EXPERIENCE_HISTORY_LAG_DAYS == FATIGUE_HISTORY_LAG_DAYS

EXPERIENCE_SUFFIXES = (
    "tourney_matches", "tourney_appearances",
    "level_matches", "level_wins", "level_losses", "level_win_pct",
)
EXPERIENCE_FEATURE_COLUMNS = [
    f"player{side}_{suffix}"
    for side in ("A", "B") for suffix in EXPERIENCE_SUFFIXES
]
REMOVED_TOURNEY_DURATION_COLUMNS = [
    f"player{side}_{suffix}"
    for side in ("A", "B")
    for suffix in ("minutes_tourney", "avg_minutes_tourney")
]


def add_experience_features(frame, history_lag_days=EXPERIENCE_HISTORY_LAG_DAYS):
    """Compute eligible experience before each tournament reference date.

    Counts are independent of ordering within an event or a reference date.
    We use the full historical input (including unknown-surface matches) but
    attach features to the filtered modeling frame using stable match_id.
    """
    required = (
        "match_id", "tourney_date", "tourney_id", "tourney_name",
        "tourney_level", "playerA_id", "playerB_id", "playerA_wins",
    )
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise KeyError(f"Missing Experience input columns: {missing}")

    source = frame.loc[:, required].copy()
    if not pd.api.types.is_datetime64_any_dtype(source["tourney_date"]):
        source["tourney_date"] = pd.to_datetime(
            source["tourney_date"], errors="raise"
        )
    if source["tourney_date"].isna().any():
        raise ValueError("Experience requires nonmissing tournament dates")
    if not source["tourney_date"].is_monotonic_increasing:
        raise ValueError("Sort by tourney_date before Experience generation")
    if source["match_id"].isna().any() or not source["match_id"].is_unique:
        raise ValueError("Experience requires unique, non-null match_id")

    tournament_matches = defaultdict(int)  # (player, tourney_name) -> match count
    tournament_editions = defaultdict(set) # (player, tourney_name) -> edition IDs
    level_record = defaultdict(lambda: [0, 0])  # (player, level) -> wins, losses
    pending = deque()  # (date, player A, player B, A won, name, level, edition)
    features = {name: [] for name in EXPERIENCE_FEATURE_COLUMNS}

    def clean_name(value):
        return str(value).strip() if pd.notna(value) and str(value).strip() else None

    def release(past):
        _, a, b, a_wins, name, level, edition = past
        # Incomplete/invalid match outcomes should not affect experience totals.
        if (pd.isna(a) or pd.isna(b) or a == b
                or pd.isna(a_wins) or a_wins not in (0, 1)):
            return
        for player, won in ((a, int(a_wins)), (b, 1 - int(a_wins))):
            if name is not None:
                key = (player, name)
                tournament_matches[key] += 1
                if edition is not None:
                    tournament_editions[key].add(edition)
            if level is not None:
                level_record[(player, level)][0 if won else 1] += 1

    previous_date = None
    cutoff = None
    for row in source.itertuples(index=False):
        date = row.tourney_date
        if previous_date is None or date != previous_date:
            cutoff = date - pd.Timedelta(days=history_lag_days)
            previous_date = date

        # Commit only dates satisfying the cutoff, BEFORE reading current features.
        # Every match with the same tournament date has the same history snapshot.
        while pending and pending[0][0] <= cutoff:
            release(pending.popleft())

        name = clean_name(row.tourney_name)
        level = clean_name(row.tourney_level)
        for side in ("A", "B"):
            player = getattr(row, f"player{side}_id")
            if pd.isna(player):
                n_tourney = n_editions = wins = losses = 0
            else:
                key = (player, name)
                n_tourney = tournament_matches[key] if name is not None else 0
                n_editions = len(tournament_editions[key]) if name is not None else 0
                wins, losses = level_record[(player, level)] if level is not None else (0, 0)
            n_level = wins + losses
            values = (
                n_tourney, n_editions, n_level, wins, losses,
                wins / n_level if n_level else 0.5,
            )
            for suffix, value in zip(EXPERIENCE_SUFFIXES, values):
                features[f"player{side}_{suffix}"].append(value)

        # The same edition-key convention is used in the Fatigue block.
        pending.append((
            date, row.playerA_id, row.playerB_id, row.playerA_wins,
            name, level, _fatigue_event_key(row),
        ))

    output = pd.DataFrame(features)
    output.insert(0, "match_id", source["match_id"].to_numpy())
    return output


# Include historical records with unknown surfaces, but never reintroduce
# those rows into the model-ready matches_df.
experience_full = add_experience_features(matches_df_fixed_copy)
matches_df.drop(columns=REMOVED_TOURNEY_DURATION_COLUMNS,
                errors="ignore", inplace=True)
_attach_historical_features(matches_df, experience_full,
                            EXPERIENCE_FEATURE_COLUMNS)
print("Attached", len(EXPERIENCE_FEATURE_COLUMNS), "Experience columns by match_id")


Attached 12 Experience columns by match_id


C:\Users\HP\AppData\Local\Temp\ipykernel_3604\4182400259.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  model_frame[feature_columns] = lookup.loc[


In [124]:
# V1.1 Experience validation — synthetic tests, no external ATP CSV required.
from pandas.testing import assert_frame_equal

_test_rows = [
    (0, "2026-01-01", "E1", "Wimbledon", "G", 1, 2, 1, np.nan),
    (1, "2026-01-01", "E1", "Wimbledon", "G", 1, 3, 0, "Grass"),
    (2, "2026-01-05", "E2", "Wimbledon", "G", 1, 4, 1, "Grass"),
    (3, "2026-01-11", "E3", "Australian Open", "G", 1, 5, 1, "Hard"),
    (4, "2026-01-15", "E4", "Wimbledon", "G", 1, 6, 1, "Grass"),
    (5, "2026-01-22", "E5", "Wimbledon", "M", 1, 7, 1, "Grass"),
    (6, "2026-02-02", "E6", "Wimbledon", "G", 1, 8, 0, "Grass"),
]
_test_experience = pd.DataFrame(_test_rows, columns=[
    "match_id", "tourney_date", "tourney_id", "tourney_name",
    "tourney_level", "playerA_id", "playerB_id", "playerA_wins", "surface",
])
_test_experience["tourney_date"] = pd.to_datetime(_test_experience["tourney_date"])
_test_result = add_experience_features(_test_experience).set_index("match_id")

# Jan 11: Jan 1 is eligible (exactly 10 days); Jan 5 is not.
assert _test_result.loc[3, "playerA_tourney_matches"] == 0  # different name
assert _test_result.loc[3, "playerA_level_matches"] == 2
assert _test_result.loc[3, "playerA_level_wins"] == 1
assert _test_result.loc[3, "playerA_level_losses"] == 1
# Jan 15: same-name Jan 1 + Jan 5 are eligible, from two editions.
assert _test_result.loc[4, "playerA_tourney_matches"] == 3
assert _test_result.loc[4, "playerA_tourney_appearances"] == 2
# Jan 22: Jan 11 eligible; Jan 15 not eligible (only 7 days old).
assert _test_result.loc[5, "playerA_level_matches"] == 0  # first Masters event
assert _test_result.loc[5, "playerA_tourney_matches"] == 3
# Feb 2: all earlier tournaments eligible; Jan 22 adds a third edition.
assert _test_result.loc[6, "playerA_tourney_matches"] == 5
assert _test_result.loc[6, "playerA_tourney_appearances"] == 4
assert _test_result.loc[6, "playerA_level_matches"] == 5
assert _test_result.loc[6, "playerA_level_wins"] == 4

# Input row order on identical dates must not change ANY generated feature.
_shuffled = pd.concat([
    group.sample(frac=1, random_state=47)
    for _, group in _test_experience.groupby("tourney_date", sort=True)
], ignore_index=True)
_shuffled_result = add_experience_features(_shuffled).set_index("match_id")
assert_frame_equal(_test_result.sort_index(), _shuffled_result.sort_index())

# Observed vs missing tournament metadata and outcome validity.
_bad = pd.DataFrame([
    (10, "2026-01-01", "E1", "Open X", "G", 9, 10, np.nan),
    (11, "2026-01-12", "E2", "Open X", "G", 9, 11, 1),
], columns=["match_id", "tourney_date", "tourney_id", "tourney_name",
            "tourney_level", "playerA_id", "playerB_id", "playerA_wins"])
_bad["tourney_date"] = pd.to_datetime(_bad["tourney_date"])
_bad_result = add_experience_features(_bad).set_index("match_id")
assert _bad_result.loc[11, "playerA_tourney_matches"] == 0
assert _bad_result.loc[11, "playerA_level_matches"] == 0
assert _bad_result.loc[11, "playerA_level_win_pct"] == 0.5

assert set(REMOVED_TOURNEY_DURATION_COLUMNS).isdisjoint(matches_df.columns)
assert matches_df[EXPERIENCE_FEATURE_COLUMNS].notna().all().all()
for _side in ("A", "B"):
    assert (matches_df[f"player{_side}_tourney_appearances"]
            <= matches_df[f"player{_side}_tourney_matches"]).all()
    assert (matches_df[f"player{_side}_level_wins"]
            + matches_df[f"player{_side}_level_losses"]
            == matches_df[f"player{_side}_level_matches"]).all()
print("Experience V1.1 synthetic validation passed")


Experience V1.1 synthetic validation passed


# **Saving the File**

In [125]:
matches_df.to_csv(DF_DIR / "matches_1991_2026_features_added.csv", index = False)

In [126]:
matches_df.info()

<class 'pandas.DataFrame'>
Index: 855077 entries, 0 to 855129
Columns: 238 entries, tourney_id to playerB_level_win_pct
dtypes: datetime64[us](1), float64(176), int64(30), int8(15), str(16)
memory usage: 1.4 GB
